# 18 · Budgeted adaptive counts with conservative error bounds

**Research question:** under the same actual serialized-byte cap, can allocating finer cells adaptively provide more
useful 25%-error certificates than a strong uniform finer histogram? This is a concrete candidate to test, not a claim
that adaptive histograms, quadtrees, error bounds or shared-template merging are new.

## Run on Databricks

1. Pull `experiments/e2e-source-information` in your personal Git folder. Open this notebook on the same compute.
2. Run the dependency and configuration cells. Normally leave **`run_dir` blank**: it selects the latest completed
   **17 `fresh-query-validation-...`** folder under your personal `synopsislake-results`. For exact reproduction,
   enter that complete result directory. Do not enter a notebook, ZIP, 15 or 16 directory.
3. Keep **seeds `0,1,2`**, **max_depth `10`**, **training_cells `4000`**, **budget_bytes `0`** for the planned run.
   Zero means inherit 15's raw2 synopsis budget (276,984 bytes for the current inputs); it does not mean no storage.
   Choose settings before looking at this experiment's quality results. Do not sweep priorities on inspected labels.
4. Leave relocation fields blank when existing result folders and raw inputs remain accessible. `source15_dir` and
   `source08d_dir` are optional relocated COMPLETE result directories with matching hashes. `raw_data_dir` points to
   unchanged `0.csv`, `1.csv`, `2.csv.gz`, `3.csv.gz`; `queries_file` and `histogram_dir` are relocation overrides.
   Matching source 08d/15 result folders and the original histogram export must remain available. No model files are loaded.
5. Run All. Raw construction, layout selection, artifact loading, merge and query costs are recorded separately.
   A small plumbing check can use max_depth=5, seeds=0 and smaller timing/training caps; this is not the planned comparison.
6. Share **`adaptive_bounds_results.zip`** from the printed new `adaptive-bounds-...` directory. Existing outputs are
   preserved. Source/merged binary summaries stay in its `summaries/` folder; the ZIP contains diagnostics, not raw points.

## Fixed comparison and scope

| Candidate | Where the extra detail goes | Information used to choose its layout |
|---|---|---|
| `uniform` | Largest square grid fitting the byte cap | Root and budget only |
| `adaptive_data` | Best-first quadtree, positive mass × 2^-depth | Exact construction-time pilot counts |
| `adaptive_relative_width` | Best-first reduction of weighted conservative query-count interval width | Same pilot and OLD outer-training query geometry only |

All candidates use the same compact uint32 exact-count codec and the **same** bound/certificate engine. Uniform
boundaries are implicit, so it does not pay for redundant per-cell coordinates. Adaptive pays for packed tree topology.
The cap includes header, counts and topology. Actual bytes, summed source-file storage, resident numeric buffers and
temporary construction allocations are reported separately. There is no model or inference cost to omit.

The data-only and uniform controls are constructed once and reused across layout seeds. Seeds identify existing
training splits, not new independent datasets. One hash-selected training rectangle per generating cell is used,
capped at 4000 cells. Relative-width weights come from a coarse pilot answer floored at10, never true query counts.
Zero-immediate-gain positive-mass splits are allowed to reach potentially useful deeper splits; ties are deterministic.

**This prototype uses coordinated ingestion.** A coordinator chooses one common template before sources are compressed.
Each source produces exact counts in that template. Merge reloads those summaries and adds their counts without raw
points or pilot access. A different template is rejected. This does **not** solve arbitrary heterogeneous ARM merging,
and the global pilot/layout-search cost is not hidden. Canonical bin boundaries are shared and recorded in the codec.

**The answers change.** This compares new exact-mass partitions, not another model evaluating the original ARM answer.
An old estimated ARM bucket mass cannot be treated as an exact raw count to obtain these guarantees. The point answers
are still approximate within partially intersected leaves. We separately measure answer quality and certificate utility.

## What “certified” means

For a query, fully contained leaf masses give a lower count **L**. Every intersecting leaf, including boundary contact,
contributes to an upper count **U**. Exact leaf masses imply L ≤ truth ≤ U. H is the area-interpolated point answer.
With L>0, H is certified at25% only if **H ≤1.25L AND H ≥0.75U**, checked by exact integer-ratio comparisons.
The zero interval H=L=U=0 is certified. If L=0<U, no certificate is issued; there is no evaluation-floor substitution.
At true count zero, actual error is0 if H=0 and infinity otherwise.

There is **no calibration, new risk threshold, model fitting or test-based layout selection**. This certificate is a
geometric worst-case rule under exact-count and fixed-root assumptions, not a learned probability. It can be valid but
too wide to be useful. Zero certified answers means undefined observed violation rate, not0%.

All 17 candidates, including zero/small counts, are checked for interval containment and false certificates.
Primary utility uses the same truth≥10 cohort from17. New generating groups and existing held-out groups stay separate.
**17 has already been inspected: these results are development evidence, not fresh final validation.** A useful advantage
would justify freezing one candidate for untouched queries/data. No useful advantage over uniform means stop this
candidate under this budget/workload, rather than tune until the inspected test looks good.

Resident query timing includes estimation, bound calculation and decision. Loading/decoding/indexing, construction and
merge are separate. Exact fallback is not run here; this is not a production E2E speedup claim. Repeated queries across
seeds are not independent observations. Broader literature review is still needed before a novelty claim.

## Dependencies

Only NumPy, pandas and Matplotlib are needed; LightGBM is not used. Install missing packages only. If the runtime restarts,
run the following configuration cell and then continue. Existing package versions are recorded, not silently replaced.

In [ ]:
import importlib.util
_missing = [name for name in ('numpy', 'pandas', 'matplotlib') if importlib.util.find_spec(name) is None]
if _missing:
    get_ipython().run_line_magic('pip', 'install ' + ' '.join(_missing))
    try:
        dbutils.library.restartPython()
    except NameError:
        pass
else:
    print('Dependencies available. No model package is required.')

## Configuration and reproducible scope

Create widgets with portable personal-workspace defaults. A blank run_dir means completed17, not15 or16.

In [ ]:
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4
from zipfile import ZipFile, ZIP_DEFLATED

import numpy as np
import pandas as pd

METHODS = ('uniform', 'adaptive_data', 'adaptive_relative_width')
FAMILIES = ('new_groups', 'existing_test_groups')
MIN_EXACT = 10
EPSILON = .25
IMPLEMENTATION_SHA256 = 'd5ea19cde07dbf60105ba9c55f4030952a6ebdf1145fefe373844a53dc8cad24'


def notebook_defaults():
    try:
        context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        root = context.notebookPath().get().split('/ml/notebooks/')[0]
        repo = root if root.startswith('/Workspace/') else '/Workspace'+root
        output = '/Workspace/Users/'+context.userName().get()+'/synopsislake-results'
    except (NameError, AttributeError):
        repo = str(Path.cwd());output = str(Path(tempfile.gettempdir())/'synopsislake-results')
    return dict(repo_root=repo, output_root=output, run_dir='', source15_dir='', source08d_dir='', raw_data_dir='',
                queries_file='', histogram_dir='', budget_bytes='0', seeds='0,1,2',
                max_depth='10', training_cells='4000', timing_queries='30', timing_repeats='5')


DEFAULTS = notebook_defaults()
try:
    for _key, _value in DEFAULTS.items():
        dbutils.widgets.text(_key, _value)
    get_config = dbutils.widgets.get
except NameError:
    def get_config(key):
        return os.environ.get(key.upper(), DEFAULTS[key])


def load_settings():
    cfg = {key: str(get_config(key)).strip() for key in DEFAULTS}
    cfg['seeds'] = [int(x) for x in cfg['seeds'].split(',')]
    if not cfg['seeds'] or len(set(cfg['seeds'])) != len(cfg['seeds']):
        raise ValueError('Use distinct existing outer-split seeds, normally 0,1,2')
    for key in ('budget_bytes', 'max_depth', 'training_cells', 'timing_queries', 'timing_repeats'):
        cfg[key] = int(cfg[key])
    if cfg['budget_bytes'] < 0 or not 1 <= cfg['max_depth'] <= 10:
        raise ValueError('budget_bytes must be >=0; max_depth must be in 1..10')
    if min(cfg['training_cells'], cfg['timing_queries'], cfg['timing_repeats']) < 1:
        raise ValueError('Training-cell cap and timing settings must be positive')
    return cfg


def sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()


def save_json(path, value):
    path = Path(path); temporary = path.with_suffix(path.suffix+'.tmp')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False, default=str), encoding='utf-8')
    temporary.replace(path)


def stable_ids(ids, n, salt):
    return sorted(sorted(set(map(str, ids)),
                         key=lambda item: hashlib.sha256(f'{salt}:{item}'.encode()).digest())[:n])

## Count-summary codec, exact common-layout merge and query bounds

Definitions only. Stored leaf masses are exact; closed query boundaries, count overflow and incompatible layouts are checked. Runtime retains persisted counts and derived indices, never the construction pilot.

In [ ]:
"""Budgeted common-layout count summaries with conservative closed-query bounds.

The adaptive template is chosen once, before source compression, from a shared
pilot count atlas and (for relative_width) training-query geometry only. Each
source then stores exact counts on that immutable partition. Merge is exact
checked count addition under the same schema; heterogeneous partitions are
rejected, never implicitly rebinned. Adaptive layout/bounds are classical
components; this module makes no novelty or performance claim.
"""
from pathlib import Path
from fractions import Fraction
import hashlib
import heapq
import math
import struct

import numpy as np

MAGIC = b'SLADP001'
VERSION = 1
HEADER = struct.Struct('<8sBBBBIIddddII4x')
assert HEADER.size == 64
UINT32_MAX = int(np.iinfo(np.uint32).max)
DIRECTORY_SIDE = 64
EPSILON = .25


def _root(root):
    values = tuple(map(float, root))
    if (len(values) != 4 or not np.isfinite(values).all() or
            not values[0] < values[1] or not values[2] < values[3] or
            not np.isfinite(values[1]-values[0]) or not np.isfinite(values[3]-values[2])):
        raise ValueError('root must be finite increasing (xmin,xmax,ymin,ymax)')
    return values


def _integer_counts(values, ndim=None):
    array = np.asarray(values)
    if ndim is not None and array.ndim != ndim:
        raise ValueError('Invalid count-array dimensions')
    if (not np.issubdtype(array.dtype, np.number) or np.iscomplexobj(array) or
            not np.isfinite(array).all() or (array < 0).any() or
            not np.equal(array, np.floor(array)).all()):
        raise ValueError('Counts must be finite nonnegative integers')
    if array.max(initial=0) > UINT32_MAX:
        raise OverflowError('A stored count exceeds uint32')
    # This also bounds exact integer certificate products in float64; actual
    # experiments have about ten million rows, far inside this precision range.
    total = sum(map(int, array.ravel()))
    if total > (2**53-1)//5:
        raise OverflowError('Total mass exceeds exact certificate precision')
    return np.asarray(array, dtype=np.uint32)


def _pilot(values):
    values = _integer_counts(values, 2)
    if values.shape[0] != values.shape[1] or values.shape[0] < 1:
        raise ValueError('Pilot must be a nonempty square power-of-two count grid')
    n = values.shape[0]
    if n & (n-1):
        raise ValueError('Pilot side must be a power of two')
    return values, int(n.bit_length()-1)


def _pyramid(values):
    levels = [np.asarray(values, dtype=np.uint64)]
    while levels[-1].shape[0] > 1:
        a = levels[-1];n = a.shape[0]
        levels.append(a.reshape(n//2,2,n//2,2).sum(axis=(1,3), dtype=np.uint64))
    return list(reversed(levels))


def _rectangles(rectangles):
    values = np.asarray(rectangles, dtype=float)
    if values.size == 0:
        return np.empty((0,4), dtype=float)
    if (values.ndim != 2 or values.shape[1] != 4 or not np.isfinite(values).all() or
            (values[:,0] > values[:,1]).any() or (values[:,2] > values[:,3]).any()):
        raise ValueError('Queries must be finite closed (xmin,xmax,ymin,ymax) rectangles')
    return values


def _dyadic_bounds(root, depth, ix, iy):
    n = 2**int(depth);x0,x1,y0,y1 = root
    dx,dy = (x1-x0)/n,(y1-y0)/n
    # np.linspace uses this power-of-two scale as well. Preserve exact outer
    # endpoints rather than recomputing them as root_min + rounded span.
    return (x0 if ix == 0 else x0+ix*dx,
            x1 if ix+1 == n else x0+(ix+1)*dx,
            y0 if iy == 0 else y0+iy*dy,
            y1 if iy+1 == n else y0+(iy+1)*dy)


def _partial_mask(rectangles, bounds):
    x0,x1,y0,y1 = bounds
    intersects = ((rectangles[:,1] >= x0) & (rectangles[:,0] <= x1) &
                  (rectangles[:,3] >= y0) & (rectangles[:,2] <= y1))
    contains = ((rectangles[:,0] <= x0) & (rectangles[:,1] >= x1) &
                (rectangles[:,2] <= y0) & (rectangles[:,3] >= y1))
    return intersects & ~contains


def _count_estimate(rectangle, counts, root):
    nx,ny = counts.shape;x0,x1,y0,y1 = root
    bx = np.linspace(x0,x1,nx+1);by = np.linspace(y0,y1,ny+1)
    q0,q1,q2,q3 = rectangle
    wx = np.maximum(0.,np.minimum(q1,bx[1:])-np.maximum(q0,bx[:-1]))/np.diff(bx)
    wy = np.maximum(0.,np.minimum(q3,by[1:])-np.maximum(q2,by[:-1]))/np.diff(by)
    return float(wx @ counts @ wy)


def _adaptive_size(leaves):
    leaves = int(leaves)
    if leaves < 1 or (leaves-1) % 3:
        raise ValueError('A full quadtree has 1+3k leaves')
    nodes = (4*leaves-1)//3
    return HEADER.size + (nodes+7)//8 + 4*leaves


def _layout_digest(kind, root, nx, ny, max_depth, topology):
    topology = np.asarray(topology, dtype=np.uint8)
    packed = np.packbits(topology, bitorder='little').tobytes()
    # Counts are intentionally excluded: this identifies geometry/schema only.
    encoded = struct.pack('<BII B dddd I', int(kind),int(nx),int(ny),int(max_depth),*root,len(topology))
    return hashlib.sha256(encoded+packed).hexdigest()


def _decode_topology(topology, root, max_depth):
    topo = np.asarray(topology)
    if topo.ndim != 1 or not len(topo) or not np.isin(topo,[0,1]).all():
        raise ValueError('Invalid adaptive preorder topology')
    root = _root(root)
    if not 0 <= int(max_depth) <= 30:
        raise ValueError('Adaptive max_depth must be in 0..30')
    stack = [(0,0,0)];position = 0;leaves = []
    while stack:
        if position >= len(topo):
            raise ValueError('Truncated adaptive topology')
        depth,ix,iy = stack.pop();internal = bool(topo[position]);position += 1
        if internal:
            if depth >= max_depth:
                raise ValueError('Topology exceeds declared maximum depth')
            for a,b in reversed(((0,0),(0,1),(1,0),(1,1))):
                stack.append((depth+1,2*ix+a,2*iy+b))
        else:
            leaves.append((depth,ix,iy))
    if position != len(topo):
        raise ValueError('Trailing adaptive topology nodes')
    if max(d for d,_,_ in leaves) != int(max_depth):
        raise ValueError('Noncanonical adaptive maximum depth')
    bounds = np.asarray([_dyadic_bounds(root,*leaf) for leaf in leaves], dtype=float)
    if (bounds[:,0] >= bounds[:,1]).any() or (bounds[:,2] >= bounds[:,3]).any():
        raise ValueError('Leaf coordinates cannot be represented at this depth')
    return leaves,bounds


class Summary:
    """Only persisted geometry and leaf counts, plus an O(leaves+4096) directory.

    No pilot grid, raw points, training rectangles, labels or split priorities
    are retained. Counts are immutable. Geometry arrays are derived at loading.
    """
    def __init__(self, kind, root, counts, nx=0, ny=0, topology=None, max_depth=0):
        self.kind = str(kind);self.root = _root(root)
        self.counts = _integer_counts(counts).reshape(-1).copy()
        if not len(self.counts):
            raise ValueError('A summary needs at least one leaf')
        self.counts.setflags(write=False)
        self.nx,self.ny = int(nx),int(ny)
        if self.kind == 'uniform':
            if self.nx < 1 or self.ny < 1 or self.nx*self.ny != len(self.counts):
                raise ValueError('Uniform dimensions do not match counts')
            self.max_depth = 0;self.topology = np.empty(0,dtype=np.uint8)
            bx = np.linspace(self.root[0],self.root[1],self.nx+1)
            by = np.linspace(self.root[2],self.root[3],self.ny+1)
            if (np.diff(bx) <= 0).any() or (np.diff(by) <= 0).any():
                raise ValueError('Uniform cell coordinates collapse in float64')
            self.leaf_bounds = np.column_stack((np.repeat(bx[:-1],self.ny),np.repeat(bx[1:],self.ny),
                np.tile(by[:-1],self.nx),np.tile(by[1:],self.nx)))
            kind_number = 0
        elif self.kind == 'adaptive':
            if self.nx != 0 or self.ny != 0:
                raise ValueError('Adaptive summaries have no uniform dimensions')
            self.max_depth = int(max_depth)
            self.topology = np.asarray(topology,dtype=np.uint8).copy()
            leaf_codes,self.leaf_bounds = _decode_topology(self.topology,self.root,self.max_depth)
            if len(leaf_codes) != len(self.counts):
                raise ValueError('Adaptive leaf counts do not match topology')
            kind_number = 1
        else:
            raise ValueError('Unknown summary kind')
        self.topology.setflags(write=False)
        self.layout_hash = _layout_digest(kind_number,self.root,self.nx,self.ny,self.max_depth,self.topology)
        self.total_count = sum(map(int,self.counts))
        self._build_directory()

    @property
    def leaf_count(self):
        return len(self.counts)

    @property
    def nleaves(self):
        return len(self.counts)

    @property
    def serialized_nbytes(self):
        return HEADER.size + (len(self.topology)+7)//8 + self.counts.nbytes

    @property
    def derived_nbytes(self):
        # All retained derived array buffers. Python object headers and allocator
        # overhead remain additional runtime overhead for any implementation.
        return int(self.leaf_bounds.nbytes+self.directory_offsets.nbytes+self.directory_ids.nbytes+
                   self.directory_x.nbytes+self.directory_y.nbytes)

    def _build_directory(self):
        side = DIRECTORY_SIDE
        self.directory_x = np.linspace(self.root[0],self.root[1],side+1)
        self.directory_y = np.linspace(self.root[2],self.root[3],side+1)
        # Each positive-area leaf is assigned to positive-area bins only.
        # Queries later inspect touching bins on BOTH sides of a boundary.
        # Generate all leaf/bin pairs in arrays, then sort them by directory
        # bin. Construction uses transient O(leaves+entries) integer buffers;
        # these are released and are not part of retained derived_nbytes.
        bounds = self.leaf_bounds
        starts_x = np.clip(np.searchsorted(self.directory_x,bounds[:,0],'right')-1,0,side-1)
        ends_x = np.clip(np.searchsorted(self.directory_x,bounds[:,1],'left')-1,0,side-1)
        starts_y = np.clip(np.searchsorted(self.directory_y,bounds[:,2],'right')-1,0,side-1)
        ends_y = np.clip(np.searchsorted(self.directory_y,bounds[:,3],'left')-1,0,side-1)
        spans_y = ends_y-starts_y+1
        entries_per_leaf = (ends_x-starts_x+1)*spans_y
        starts = np.r_[0,np.cumsum(entries_per_leaf,dtype=np.int64)]
        entries = int(starts[-1])
        # Dyadic adaptive leaves finer than depth6 each touch one directory bin;
        # coarser disjoint leaves together cover at most4096. Uniform rectangles
        # have the analogous constant-factor bound even at non-dyadic N.
        if entries > 16*(self.leaf_count+side*side):
            raise ValueError('Spatial directory violates its bounded-overhead design')
        leaf_ids = np.repeat(np.arange(self.leaf_count,dtype=np.uint32),entries_per_leaf)
        within_leaf = np.arange(entries,dtype=np.int64)-np.repeat(starts[:-1],entries_per_leaf)
        repeated_spans_y = np.repeat(spans_y,entries_per_leaf)
        bin_ids = (np.repeat(starts_x,entries_per_leaf)+within_leaf//repeated_spans_y)*side
        bin_ids += np.repeat(starts_y,entries_per_leaf)+within_leaf%repeated_spans_y
        order = np.argsort(bin_ids,kind='stable')
        lengths = np.bincount(bin_ids,minlength=side*side)
        self.directory_offsets = np.r_[0,np.cumsum(lengths,dtype=np.int64)]
        self.directory_ids = leaf_ids[order]
        # Named temporary array buffers only, excluding NumPy's internal sort
        # workspace and short-lived expression temporaries. This diagnostic
        # is not presented as a process peak-memory measurement.
        self.directory_construction_named_nbytes = int(sum(array.nbytes for array in
            (starts_x,ends_x,starts_y,ends_y,spans_y,entries_per_leaf,starts,
             leaf_ids,within_leaf,repeated_spans_y,bin_ids,order,lengths)))

    def _candidate_ids(self, q):
        side = DIRECTORY_SIDE
        i0 = int(np.clip(np.searchsorted(self.directory_x,q[0],'left')-1,0,side-1))
        i1 = int(np.clip(np.searchsorted(self.directory_x,q[1],'right')-1,0,side-1))
        j0 = int(np.clip(np.searchsorted(self.directory_y,q[2],'left')-1,0,side-1))
        j1 = int(np.clip(np.searchsorted(self.directory_y,q[3],'right')-1,0,side-1))
        pieces = []
        for i in range(i0,i1+1):
            for j in range(j0,j1+1):
                k = i*side+j
                lo,hi = self.directory_offsets[k:k+2]
                if hi > lo:
                    pieces.append(self.directory_ids[lo:hi])
        return np.unique(np.concatenate(pieces)) if pieces else np.empty(0,dtype=np.uint32)


def uniform_summary(counts, root):
    array = _integer_counts(counts,2)
    if min(array.shape,default=0) < 1:
        raise ValueError('Uniform grid must be nonempty')
    return Summary('uniform',root,array.ravel(),nx=array.shape[0],ny=array.shape[1])


def build_layout(pilot_counts, root, training_rectangles, budget_bytes,
                 strategy='relative_width', max_depth=10, weights=None):
    """Best-first common adaptive layout under the ACTUAL binary byte cap.

    relative_width optimizes geometry-only weighted count-interval width
    reduction. Default weights use a coarse64x64 pilot answer, floored at10,
    never query truth. data prioritizes mass*2**(-depth). Positive-mass zero-gain
    candidates can still be split, because a zero immediate gain can enable a
    useful deeper split; ties are resolved deterministically by Morton order.
    Zero-mass leaves are never split. No outcome or test-query score is used.
    """
    counts,pilot_depth = _pilot(pilot_counts);root = _root(root)
    if int(budget_bytes) != budget_bytes or budget_bytes < _adaptive_size(1):
        raise ValueError('Budget cannot store an adaptive root leaf')
    if int(max_depth) != max_depth or not 0 <= max_depth <= 30:
        raise ValueError('max_depth must be in0..30')
    limit = min(int(max_depth),pilot_depth)
    if strategy not in ('relative_width','data'):
        raise ValueError('strategy must be relative_width or data')
    q = _rectangles(np.empty((0,4)) if training_rectangles is None else training_rectangles)
    if strategy == 'relative_width' and not len(q):
        raise ValueError('relative_width requires training query geometry')
    levels = _pyramid(counts)
    if weights is None:
        if strategy == 'relative_width':
            coarse = levels[min(6,pilot_depth)]
            weights = np.asarray([1/max(_count_estimate(rect,coarse,root),10.) for rect in q],float)
        else:
            weights = np.ones(len(q),float)
    else:
        weights = np.asarray(weights,float)
        if weights.shape != (len(q),) or not np.isfinite(weights).all() or (weights < 0).any():
            raise ValueError('One finite nonnegative weight per training rectangle is required')
    if len(weights) and weights.sum() > 0:
        weights = weights/weights.sum()
    # Construction-only arrays: depth,ix,iy,mass,partial_query_ids,children.
    nodes = [dict(depth=0,ix=0,iy=0,mass=int(levels[0][0,0]),
                  partial=np.flatnonzero(_partial_mask(q,root)),children=None)]
    heap = []
    def prepare(index):
        node = nodes[index];d,ix,iy = node['depth'],node['ix'],node['iy']
        if d >= limit or node['mass'] == 0:
            return
        partial = node['partial'];children = []
        after = 0.
        for a,b in ((0,0),(0,1),(1,0),(1,1)):
            ci,cj = 2*ix+a,2*iy+b
            child_ids = partial[_partial_mask(q[partial],_dyadic_bounds(root,d+1,ci,cj))]
            mass = int(levels[d+1][ci,cj])
            children.append(dict(depth=d+1,ix=ci,iy=cj,mass=mass,partial=child_ids,children=None))
            after += mass*float(weights[child_ids].sum())
        if strategy == 'relative_width':
            gain = node['mass']*float(weights[partial].sum())-after
            tolerance = 1e-10*max(1.,node['mass']*float(weights[partial].sum()))
            if gain < -tolerance:
                raise ValueError('A split increased conservative interval width')
            priority = max(gain,0.)
        else:
            priority = node['mass']*2.**(-d)
        node['candidate_children'] = children
        # Fixed-depth Morton spatial address; depth/index complete deterministic ties.
        morton = 0
        for bit in range(d-1,-1,-1):
            morton = (morton<<2) | (((ix>>bit)&1)<<1) | ((iy>>bit)&1)
        morton <<= 2*(limit-d)
        heapq.heappush(heap,(-float(priority),morton,d,index))
    prepare(0);leaves = 1;splits = 0
    while heap and _adaptive_size(leaves+3) <= budget_bytes:
        _,_,_,index = heapq.heappop(heap)
        node = nodes[index]
        if node['children'] is not None:
            continue
        children = node.pop('candidate_children')
        node['children'] = list(range(len(nodes),len(nodes)+4));nodes.extend(children)
        # Its query list no longer needs to be retained after child candidates exist.
        node['partial'] = np.empty(0,dtype=np.int64)
        leaves += 3;splits += 1
        for child in node['children']:
            prepare(child)
    topology = [];pending = [0];actual_depth = 0
    while pending:
        index = pending.pop();node = nodes[index]
        internal = node['children'] is not None;topology.append(int(internal))
        if internal:
            pending.extend(reversed(node['children']))
        else:
            actual_depth = max(actual_depth,node['depth'])
    topology = np.asarray(topology,dtype=np.uint8)
    selected, _ = _decode_topology(topology,root,actual_depth)
    if len(selected) != leaves:
        raise ValueError('Adaptive leaf accounting mismatch')
    layout = dict(kind='adaptive',root=root,max_depth=actual_depth,topology=topology,
                  layout_hash=_layout_digest(1,root,0,0,actual_depth,topology),
                  strategy=strategy,leaf_count=leaves,serialized_bytes=_adaptive_size(leaves),
                  unused_budget_bytes=int(budget_bytes)-_adaptive_size(leaves),
                  requested_max_depth=int(max_depth),selected_splits=splits,
                  pilot_pyramid_nbytes=int(sum(level.nbytes for level in levels)),
                  layout_search_memory_scope='Fixed uint64 count-pyramid buffers only; excludes node/heap/query-list objects and transient arrays; not peak RSS')
    # Deliberately no pilot counts, unselected candidate nodes, query arrays or
    # priorities are retained in the returned common template.
    return layout


def source_summary(pilot_counts, layout):
    values,depth = _pilot(pilot_counts)
    if layout.get('kind','adaptive') != 'adaptive':
        raise ValueError('source_summary requires a common adaptive layout')
    root = _root(layout['root'])
    topology = np.asarray(layout['topology'],dtype=np.uint8)
    codes,_ = _decode_topology(topology,root,int(layout['max_depth']))
    if max(d for d,_,_ in codes) > depth:
        raise ValueError('Source pilot cannot represent this shared layout')
    levels = _pyramid(values)
    counts = np.asarray([levels[d][i,j] for d,i,j in codes],dtype=np.uint64)
    if counts.max(initial=0) > UINT32_MAX:
        raise OverflowError('A selected source leaf exceeds uint32')
    summary = Summary('adaptive',root,counts,topology=topology,max_depth=layout['max_depth'])
    if summary.layout_hash != layout['layout_hash']:
        raise ValueError('Common adaptive layout hash mismatch')
    return summary


def merge_summaries(summaries, budget_bytes):
    summaries = list(summaries)
    if not summaries:
        raise ValueError('At least one source summary is required')
    first = summaries[0]
    if not isinstance(first,Summary) or any(not isinstance(s,Summary) or s.layout_hash != first.layout_hash for s in summaries):
        raise ValueError('Incompatible summary schemas; heterogeneous partitions are not rebinned')
    if int(budget_bytes) != budget_bytes or budget_bytes < HEADER.size:
        raise ValueError('budget_bytes must be a positive integer codec budget')
    total = np.zeros(first.leaf_count,dtype=np.uint64)
    for source in summaries:
        total += source.counts.astype(np.uint64)
        if total.max(initial=0) > UINT32_MAX:
            raise OverflowError('Merged leaf count exceeds uint32')
    merged = Summary(first.kind,first.root,total,nx=first.nx,ny=first.ny,
                     topology=first.topology,max_depth=first.max_depth)
    if merged.serialized_nbytes > int(budget_bytes):
        raise ValueError('Merged artifact exceeds the actual serialized budget')
    return merged


def save_summary(path, summary):
    if not isinstance(summary,Summary):
        raise TypeError('Expected a Summary')
    kind = 0 if summary.kind == 'uniform' else 1
    header = HEADER.pack(MAGIC,VERSION,kind,4,summary.max_depth,summary.nx,summary.ny,
                         *summary.root,summary.leaf_count,len(summary.topology))
    packed = np.packbits(summary.topology,bitorder='little').tobytes()
    payload = header+packed+np.asarray(summary.counts,dtype='<u4').tobytes(order='C')
    if len(payload) != summary.serialized_nbytes:
        raise ValueError('Serialized summary length mismatch')
    path = Path(path)
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_bytes(payload)
    return len(payload)


def load_summary(path):
    payload = Path(path).read_bytes()
    if len(payload) < HEADER.size:
        raise ValueError('Truncated summary header')
    magic,version,kind,dtype,max_depth,nx,ny,x0,x1,y0,y1,leaves,bits = HEADER.unpack_from(payload)
    if magic != MAGIC or version != VERSION or kind not in (0,1) or dtype != 4 or leaves < 1:
        raise ValueError('Invalid summary header')
    nbytes = (bits+7)//8
    if len(payload) != HEADER.size+nbytes+4*leaves:
        raise ValueError('Invalid summary artifact length')
    packed = np.frombuffer(payload,dtype=np.uint8,count=nbytes,offset=HEADER.size)
    expanded = np.unpackbits(packed,bitorder='little')
    if expanded[bits:].any():
        raise ValueError('Noncanonical unused topology bits')
    topology = expanded[:bits]
    counts = np.frombuffer(payload,dtype='<u4',count=leaves,offset=HEADER.size+nbytes)
    if kind == 0:
        if bits or max_depth or nx < 1 or ny < 1 or nx*ny != leaves:
            raise ValueError('Invalid uniform header')
        return Summary('uniform',(x0,x1,y0,y1),counts,nx=nx,ny=ny)
    if nx or ny or bits != (4*leaves-1)//3 or (leaves-1)%3:
        raise ValueError('Invalid adaptive header')
    return Summary('adaptive',(x0,x1,y0,y1),counts,topology=topology,max_depth=max_depth)


def _outward_float(value):
    result = float(value)
    if np.isfinite(result) and Fraction.from_float(result) < value:
        result = float(np.nextafter(result,np.inf))
    return result


def query_summary(summary, q):
    q = _rectangles(np.asarray(q,float).reshape(1,-1))[0]
    if not isinstance(summary,Summary):
        raise TypeError('Expected a persisted-count Summary')
    x0,x1,y0,y1 = summary.root
    if summary.total_count == 0 or q[1] < x0 or q[0] > x1 or q[3] < y0 or q[2] > y1:
        return dict(estimate=0.,lower=0,upper=0,worst_relative_error=0.,certified=True,
                    interval_width=0,touched_leaves=0,fully_contained_leaves=0)
    if q[0] <= x0 and x1 <= q[1] and q[2] <= y0 and y1 <= q[3]:
        count = summary.total_count
        return dict(estimate=float(count),lower=count,upper=count,worst_relative_error=0.,certified=True,
                    interval_width=0,touched_leaves=summary.leaf_count,fully_contained_leaves=summary.leaf_count)
    ids = summary._candidate_ids(q)
    bounds = summary.leaf_bounds[ids]
    touched = ((bounds[:,0] <= q[1]) & (bounds[:,1] >= q[0]) &
               (bounds[:,2] <= q[3]) & (bounds[:,3] >= q[2]))
    ids,bounds = ids[touched],bounds[touched]
    mass = summary.counts[ids]
    full = ((q[0] <= bounds[:,0]) & (bounds[:,1] <= q[1]) &
            (q[2] <= bounds[:,2]) & (bounds[:,3] <= q[3]))
    lower = int(np.sum(mass[full],dtype=np.uint64))
    upper = int(np.sum(mass,dtype=np.uint64))
    width = np.maximum(0.,np.minimum(q[1],bounds[:,1])-np.maximum(q[0],bounds[:,0]))/(bounds[:,1]-bounds[:,0])
    height = np.maximum(0.,np.minimum(q[3],bounds[:,3])-np.maximum(q[2],bounds[:,2]))/(bounds[:,3]-bounds[:,2])
    estimate = float(np.sum(np.clip(width*height,0.,1.)*mass,dtype=np.float64))
    estimate = min(max(estimate,float(lower)),float(upper))
    if upper == 0:
        worst,certified = 0.,True
    elif lower == 0:
        worst,certified = (1. if estimate == 0 else np.inf),False
    else:
        h = Fraction.from_float(estimate)
        worst = _outward_float(max(abs(h-lower)/lower,abs(h-upper)/upper))
        numerator,denominator = estimate.as_integer_ratio()
        certified = bool(4*numerator <= 5*lower*denominator and 4*numerator >= 3*upper*denominator)
    return dict(estimate=estimate,lower=lower,upper=upper,worst_relative_error=float(worst),certified=certified,
                interval_width=upper-lower,touched_leaves=len(ids),fully_contained_leaves=int(full.sum()))

## Frozen input identities and one-pass raw construction

Verify17/15 identities and original geometry. One raw pass builds the dyadic pilot and the non-dyadic uniform grid directly; no resampling is called an exact count.

In [ ]:
"""Verified 17/15 inputs and exact, one-pass raw count construction for 18.

Only old training geometry is returned for layout selection. Fresh labels are
returned separately for offline validation; they are not construction inputs.
No model, fitted feature provider or training entry point is loaded.
"""
from pathlib import Path
import ast
import base64
import gzip
import hashlib
import json
import math
import struct
import time

import numpy as np
import pandas as pd


def input_sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()


def _input_json(path):
    return json.loads(Path(path).read_text())


def _input_verified(path, digest, description):
    path = Path(path).expanduser()
    if not path.is_file():
        raise FileNotFoundError('Missing '+description+': '+str(path))
    if input_sha(path) != digest:
        raise ValueError('SHA256 mismatch for '+description+': '+str(path))
    return path.resolve()


def _input_run_dir(cfg):
    entered = str(cfg.get('run_dir', '')).strip()
    if entered:
        path = Path(entered).expanduser()
        if not (path/'run.json').is_file():
            raise FileNotFoundError('run_dir must be a completed 17 fresh-query-validation directory')
        return path.resolve()
    candidates = []
    root = Path(cfg['output_root']).expanduser()
    for file in root.glob('fresh-query-validation-*/run.json'):
        try:
            record = _input_json(file)
            if record.get('status') == 'complete' and record.get('study') == 'fresh-query-validation':
                candidates.append(file)
        except (OSError, ValueError):
            continue
    if not candidates:
        raise FileNotFoundError('No completed 17 run under output_root. Set run_dir to its full directory')
    return max(candidates, key=lambda p: p.stat().st_mtime_ns).parent.resolve()


def _input_parent(original, digest, cfg, description, override=''):
    choices = []
    if override:
        choices.append(Path(override).expanduser()/'run.json')
    else:
        choices.append(Path(original)/'run.json')
        choices.extend(Path(cfg['output_root']).expanduser().glob('*/run.json'))
    for file in choices:
        if file.is_file() and input_sha(file) == digest:
            return file.parent.resolve(), _input_json(file)
    raise FileNotFoundError('Missing matching '+description+' manifest. Preserve its original result folder or set its relocation override')


def _input_hist_loader(repo):
    # Reuse the existing binary codec definitions only. No notebook entry point
    # or training/preparation code is executed. The whole definition cell is
    # pinned before selecting its two harmless reader/validator definitions.
    file = Path(repo)/'ml/notebooks/12_compact_metadata_ablation.ipynb'
    try:
        raw=file.read_bytes();document=json.loads(raw);method='workspace filesystem'
    except (OSError,ValueError,UnicodeError):
        if not str(file).startswith('/Workspace/'):
            raise
        from databricks.sdk import WorkspaceClient
        from databricks.sdk.service.workspace import ExportFormat
        client=WorkspaceClient();last=None
        for candidate in (str(file)[len('/Workspace'):],str(file)[len('/Workspace'):].removesuffix('.ipynb')):
            try:
                exported=client.workspace.export(path=candidate,format=ExportFormat.JUPYTER)
                raw=base64.b64decode(exported.content);document=json.loads(raw)
                method='Workspace API using runtime identity';break
            except Exception as error:
                last=error
        else:
            raise ValueError('Cannot read notebook12; check repo_root and workspace permissions: '+str(last)) from last
    digest='1ac8d26a90d6e402c1223960b898fee17f79e9cea33c7656a7b699c6b30da8d3'
    matches=[]
    for cell in document['cells']:
        if cell.get('cell_type')!='code':
            continue
        try:
            candidate=ast.parse(''.join(cell['source']))
        except SyntaxError:
            continue
        if hashlib.sha256(ast.dump(candidate,include_attributes=False).encode()).hexdigest()==digest:
            matches.append(candidate)
    if len(matches)!=1:
        raise ValueError('Notebook12 histogram codec definitions changed/missing; review compatibility before running18')
    tree=matches[0]
    names = {'validate_hist', 'load_hist_payload'}
    selected = ast.Module(body=[node for node in tree.body if isinstance(node, ast.FunctionDef) and node.name in names], type_ignores=[])
    if {node.name for node in selected.body} != names:
        raise ValueError('Missing inherited histogram reader definitions')
    namespace = dict(np=np, Path=Path, _HIST_MAGIC=b'SLHIST01', _HIST_HEADER=struct.Struct('<IId'))
    exec(compile(selected, str(file)+'::readers', 'exec'), namespace)
    return namespace['load_hist_payload'], dict(path=str(file),read_method=method,document_sha256=hashlib.sha256(raw).hexdigest(),
                                               definition_ast_sha256=digest)


def load_inputs(cfg, destination, state):
    """Read immutable identities/geometry; never fit or initialize a model."""
    tick = time.perf_counter()
    destination = Path(destination)
    source17 = _input_run_dir(cfg)
    prior17 = _input_json(source17/'run.json')
    if prior17.get('status') != 'complete' or prior17.get('study') != 'fresh-query-validation':
        raise ValueError('run_dir must identify completed notebook 17, not 15/16 or an old diagnostic')
    required17 = {
        'new_queries.csv': prior17['new_query_labels_sha256'],
        'protocol_lock.json': prior17['protocol_sha256'],
        'query_plan.csv': prior17['query_plan_sha256'],
        'query_membership.csv': prior17['query_membership_sha256']}
    for name, digest in required17.items():
        _input_verified(source17/name, digest, 'frozen17 '+name)
    protocol = _input_json(source17/'protocol_lock.json')
    if protocol['query_plan_sha256'] != required17['query_plan.csv'] or protocol['membership_sha256'] != required17['query_membership.csv']:
        raise ValueError('17 manifest and protocol disagree about geometry/membership')
    labels = pd.read_csv(source17/'new_queries.csv', float_precision='round_trip')
    membership = pd.read_csv(source17/'query_membership.csv')
    plan = pd.read_csv(source17/'query_plan.csv', float_precision='round_trip')
    if labels.query_id.duplicated().any() or plan.query_id.duplicated().any() or membership.duplicated(['seed','query_id']).any():
        raise ValueError('Duplicate fresh query or membership identity')
    geometry = ['query_id','family','cell','x0','x1','y0','y1']
    if not labels[geometry].reset_index(drop=True).equals(plan[geometry].reset_index(drop=True)):
        raise ValueError('17 labelled geometry differs from the frozen pre-label plan')
    truth = labels.truth.to_numpy(float)
    rectangles = labels[['x0','x1','y0','y1']].to_numpy(float)
    if (not np.isfinite(truth).all() or (truth < 0).any() or not np.equal(truth,np.floor(truth)).all()
            or not np.isfinite(rectangles).all() or (rectangles[:,0] > rectangles[:,1]).any()
            or (rectangles[:,2] > rectangles[:,3]).any()):
        raise ValueError('Invalid exact count or rectangle in saved17 labels')
    def saved_bool(column):
        values = labels[column].astype(str).str.lower()
        if not values.isin(['true','false']).all():
            raise ValueError('Invalid saved17 boolean: '+column)
        return values.eq('true').to_numpy()
    if not np.array_equal(saved_bool('eligible'),truth>=10) or not np.array_equal(saved_bool('zero_truth'),truth==0):
        raise ValueError('Saved17 eligibility/zero flags changed')
    labels['eligible'] = truth >= 10
    labels['zero_truth'] = truth == 0
    if not set(membership.query_id).issubset(set(labels.query_id)):
        raise ValueError('17 membership contains unknown queries')
    linked = membership.merge(labels[['query_id','family','cell']],on='query_id',validate='many_to_one',suffixes=('_member','_label'))
    if not linked.family_member.equals(linked.family_label) or not linked.cell_member.equals(linked.cell_label):
        raise ValueError('17 family/cell membership identity mismatch')
    requested = list(map(int,cfg['seeds']))
    if not requested or len(set(requested)) != len(requested) or not set(requested).issubset(set(membership.seed)):
        raise ValueError('Requested seeds are absent/duplicated in saved17 memberships')
    membership = membership[membership.seed.isin(requested)].copy()
    source15, prior15 = _input_parent(prior17['source15_run'], prior17['source15_manifest_sha256'], cfg,
                                     'source15', str(cfg.get('source15_dir','')).strip())
    for name in ('test_predictions.csv','outer_splits.csv','raw_input_manifest.json','payloads.csv'):
        _input_verified(source15/name, prior17['source15_artifact_sha256'][name], 'source15 '+name)
    outer = pd.read_csv(source15/'outer_splits.csv')
    if 'split_kind' in outer and set(outer.split_kind) != {'cell'}:
        raise ValueError('18 requires the saved cell-group outer split')
    outer = outer[outer.seed.isin(requested)].copy()
    if outer.duplicated(['seed','qId']).any():
        raise ValueError('Duplicate old outer role assignments')
    for seed in requested:
        part = outer[outer.seed==seed]
        if set(part.split) != {'train','calibration','test'}:
            raise ValueError('Incomplete original outer roles for seed '+str(seed))
        sets = [set(part.loc[part.split==role,'cell']) for role in ('train','calibration','test')]
        if any(sets[i]&sets[j] for i in range(3) for j in range(i)):
            raise ValueError('Original outer generating cells overlap between roles')
    # New-group IDs must stay completely outside the old fitting/calibration/
    # test inventory; supplement IDs must remain that seed's held-out groups.
    all_outer = pd.read_csv(source15/'outer_splits.csv')
    if set(membership.loc[membership.family=='new_groups','cell']) & set(all_outer.cell):
        raise ValueError('Saved17 primary group unexpectedly overlaps an old outer group')
    for seed in requested:
        supplement = set(membership.loc[(membership.seed==seed)&(membership.family=='existing_test_groups'),'cell'])
        heldout = set(outer.loc[(outer.seed==seed)&(outer.split=='test'),'cell'])
        if not supplement.issubset(heldout):
            raise ValueError('Saved17 supplement is not the corresponding seed held-out group')
    query_item = prior17['verified_inputs']['data/queries-Ans.txt']
    query_path = _input_verified(str(cfg.get('queries_file','')).strip() or query_item['path'], query_item['sha256'], 'original training query geometry')
    old = pd.read_csv(query_path,header=None).to_numpy(float)  # original parser retained intentionally
    if old.ndim!=2 or old.shape[1]!=5 or not np.isfinite(old).all():
        raise ValueError('Original query file must contain four coordinates and a count')
    if outer.qId.min()<0 or outer.qId.max()>=len(old):
        raise ValueError('Old outer IDs lie outside original query file')
    old_queries = old[:,:4].copy()  # never return old ground truth to layout selection
    del old
    saved = pd.read_csv(source15/'test_predictions.csv',usecols=['histogram_key','base_rate'])
    pairs = saved.drop_duplicates()
    if len(pairs)!=1:
        raise ValueError('18 currently requires one fixed original merged histogram/rate; do not select by new quality')
    base_rate = float(pairs.iloc[0].base_rate);key = str(pairs.iloc[0].histogram_key)
    table = pd.read_csv(source15/'payloads.csv')
    row = table[(table.variant=='raw2_source')&np.isclose(table.base_rate,base_rate,rtol=0,atol=1e-12)]
    if len(row)!=1:
        raise ValueError('Missing unique raw2 byte budget in15 payload table')
    inherited_budget = int(row.iloc[0].total_synopsis_bytes)
    entered_budget = int(cfg.get('budget_bytes',0) or 0)
    budget = entered_budget if entered_budget else inherited_budget
    if budget < 80:
        raise ValueError('Budget must fit the common64-byte header and at least one refinement')
    source08d, original = _input_parent(prior15['source_run'],prior15['source_manifest_sha256'],cfg,'original08d',
                                      str(cfg.get('source08d_dir','')).strip())
    export_key = 'run/'+key+'.txt'
    manifest_file = source15/'preparation/input_manifest.json'
    manifest = _input_json(manifest_file) if manifest_file.is_file() else {}
    export_item = manifest.get(export_key,dict(path=str(source08d/'inputs'/export_key),sha256=original['input_sha256'][export_key]))
    if export_item['sha256'] != original['input_sha256'][export_key]:
        raise ValueError('15 staged histogram identity differs from original08d')
    export_path = Path(cfg['histogram_dir'])/(key+'.txt') if str(cfg.get('histogram_dir','')).strip() else Path(export_item['path'])
    export_path = _input_verified(export_path,export_item['sha256'],'original merged histogram export')
    lines = export_path.read_text().splitlines()
    bx = np.asarray(lines[1].split(','),float);by = np.asarray(lines[2].split(','),float)
    if not np.isfinite(np.r_[bx,by]).all() or (np.diff(bx)<=0).any() or (np.diff(by)<=0).any():
        raise ValueError('Invalid original merged histogram bounds')
    loader, reuse = _input_hist_loader(cfg['repo_root'])
    binary_path = source15/'preparation/payloads'/(key+'.hist')
    if binary_path.is_file():
        hist = loader(binary_path)
        export_counts = np.asarray([line.split(',') for line in lines[3:]],float)
        if not np.array_equal(hist[1],bx) or not np.array_equal(hist[2],by) or not np.array_equal(hist[3],export_counts):
            raise ValueError('15 binary merged histogram differs from verified original export')
        if binary_path.stat().st_size != int(row.iloc[0].base_histogram_bytes):
            raise ValueError('Original merged histogram byte budget changed')
    raw_manifest = _input_json(source15/'raw_input_manifest.json')
    if not raw_manifest or len({str(r['source_id']) for r in raw_manifest}) != len(raw_manifest):
        raise ValueError('Missing/duplicate raw sources')
    parity = pd.read_csv(source15/'raw_source_parity.csv')
    if parity.source_id.duplicated().any() or not parity.subcell_mass_conserved.astype(str).str.lower().eq('true').all() or not parity.max_coarse_count_gap.eq(0).all():
        raise ValueError('Source15 exact-raw/coarse parity did not pass')
    parity = parity.set_index(parity.source_id.astype(str))
    resolved = []
    for record in raw_manifest:
        source_id = str(record['source_id']);name=Path(record['raw_path']).name
        choices = [Path(cfg['raw_data_dir'])/name] if str(cfg.get('raw_data_dir','')).strip() else [Path(record['raw_path']),Path(cfg['repo_root'])/'range_query'/name]
        path = next((p for p in choices if p.is_file()),None)
        if path is None:
            raise FileNotFoundError('Matching raw data missing: '+name+'. Set raw_data_dir to the unchanged raw-file folder')
        expected = int(parity.loc[source_id,'raw_rows'])
        if expected < 0:
            raise ValueError('Invalid expected raw row count')
        resolved.append(dict(record,raw_path=str(path.resolve()),expected_rows=expected))
    if int(row.iloc[0].auxiliary_bytes) != sum(int(r['payload_bytes']) for r in raw_manifest):
        raise ValueError('Inherited raw2 auxiliary budget does not match frozen payload sizes')
    if inherited_budget != int(row.iloc[0].base_histogram_bytes)+int(row.iloc[0].auxiliary_bytes):
        raise ValueError('Inherited raw2 total byte budget is inconsistent')
    if sum(r['expected_rows'] for r in resolved)>np.iinfo(np.uint32).max:
        raise OverflowError('This prototype codec requires the entire row count to fit uint32')
    outer.to_csv(destination/'outer_splits.csv',index=False)
    labels.to_csv(destination/'new_queries.csv',index=False,float_format='%.17g')
    membership.to_csv(destination/'query_membership.csv',index=False)
    state.update(source17_run=str(source17),source17_manifest_sha256=input_sha(source17/'run.json'),
                 source17_artifact_sha256=required17,source15_run=str(source15),
                 source15_manifest_sha256=input_sha(source15/'run.json'),
                 source08d_run=str(source08d),source08d_manifest_sha256=input_sha(source08d/'run.json'),
                 old_query_file=dict(path=str(query_path),sha256=query_item['sha256']),
                 original_histogram=dict(key=key,path=str(export_path),sha256=export_item['sha256'],base_rate=base_rate),
                 inherited_raw2_budget_bytes=inherited_budget,budget_bytes=budget,reuse_histogram_codec=reuse)
    state.setdefault('timings',{})['input_validation_seconds']=time.perf_counter()-tick
    return dict(labels=labels,membership=membership,old_queries=old_queries,outer=outer,
                root=(float(bx[0]),float(bx[-1]),float(by[0]),float(by[-1])),raw_manifest=resolved,
                budget_bytes=budget,base_rate=base_rate)


def iter_count_raw_points(path,chunk_size=100000):
    # Identical semantics to15 iter_raw_points: first two TAB fields; round-trip
    # float parser; every row counts once, including duplicates; gzip allowed.
    opener = gzip.open if str(path).endswith('.gz') else open
    with opener(path,'rt',encoding='utf-8') as stream:
        for part in pd.read_csv(stream,sep='\t',header=None,usecols=[0,1],dtype=float,
                                chunksize=chunk_size,float_precision='round_trip',skip_blank_lines=False):
            points=part.to_numpy(float)
            if points.shape[1]!=2 or not np.isfinite(points).all():
                raise ValueError('Invalid/nonfinite raw coordinates in '+str(path))
            yield points


def construct_count_inputs(data,max_depth,uniform_n,destination,state):
    """One raw pass constructs both exact dyadic pilots and non-dyadic grids.

    Uniform263 is counted directly. Resampling a1024 pilot would lose exactness
    and invalidate the subsequent lower/upper count guarantees.
    """
    max_depth=int(max_depth);uniform_n=int(uniform_n)
    if not 0<=max_depth<=12 or uniform_n<1:
        raise ValueError('Pilot max_depth must be in0..12 and uniform_n positive')
    side=2**max_depth;x0,x1,y0,y1=map(float,data['root'])
    pilot_edges=(np.linspace(x0,x1,side+1),np.linspace(y0,y1,side+1))
    uniform_edges=(np.linspace(x0,x1,uniform_n+1),np.linspace(y0,y1,uniform_n+1))
    if any((np.diff(edges)<=0).any() for edges in (*pilot_edges,*uniform_edges)):
        raise ValueError('Requested count-grid boundaries collapse in float64')
    total_tick=time.perf_counter();sources=[];costs=[]
    persistent_numeric=0;peak_numeric=0
    for record in data['raw_manifest']:
        path=Path(record['raw_path']);tick=time.perf_counter()
        _input_verified(path,record['raw_sha256'],'raw source'+str(record['source_id']))
        hash_before_seconds=time.perf_counter()-tick
        pilot=np.zeros((side,side),np.uint32);uniform=np.zeros((uniform_n,uniform_n),np.uint32)
        rows=0;scan_tick=time.perf_counter();largest_chunk=0
        for points in iter_count_raw_points(path):
            rows+=len(points);largest_chunk=max(largest_chunk,len(points))
            if ((points[:,0]<x0)|(points[:,0]>x1)|(points[:,1]<y0)|(points[:,1]>y1)).any():
                raise ValueError('Raw point lies outside the inherited common root; cannot certify dropped points')
            for counts,edges,n in ((pilot,pilot_edges,side),(uniform,uniform_edges,uniform_n)):
                ix=np.minimum(np.searchsorted(edges[0],points[:,0],side='right')-1,n-1)
                iy=np.minimum(np.searchsorted(edges[1],points[:,1],side='right')-1,n-1)
                # Internal ownership is half-open; the root's outer maximum
                # belongs to its last cell. Bounds queries remain closed.
                additions=np.bincount(ix*n+iy,minlength=n*n).reshape(n,n)
                summed=counts.astype(np.uint64)+additions.astype(np.uint64)
                if summed.max(initial=0)>np.iinfo(np.uint32).max:
                    raise OverflowError('Exact source-cell mass exceeds uint32 codec')
                counts[:]=summed
                # Deliberately report a conservative upper bound for live
                # numeric arrays; this is not an RSS/process-memory measure.
                numeric=(persistent_numeric+pilot.nbytes+uniform.nbytes+points.nbytes+ix.nbytes+iy.nbytes+
                         additions.nbytes+summed.nbytes+max(pilot.size,uniform.size)*16+len(points)*16)
                peak_numeric=max(peak_numeric,int(numeric))
            del points,ix,iy,additions,summed
        scan_seconds=time.perf_counter()-scan_tick
        tick=time.perf_counter();_input_verified(path,record['raw_sha256'],'raw source after construction')
        hash_after_seconds=time.perf_counter()-tick
        if rows!=int(record['expected_rows']) or int(pilot.sum(dtype=np.uint64))!=rows or int(uniform.sum(dtype=np.uint64))!=rows:
            raise ValueError('Raw row count or exact count-grid conservation mismatch for source'+str(record['source_id']))
        sources.append(dict(source_id=str(record['source_id']),pilot=pilot,uniform=uniform))
        persistent_numeric+=pilot.nbytes+uniform.nbytes
        costs.append(dict(source_id=str(record['source_id']),raw_path=str(path),raw_sha256=record['raw_sha256'],
                          raw_file_bytes=path.stat().st_size,raw_rows=rows,pilot_side=side,uniform_side=uniform_n,
                          pilot_array_bytes=pilot.nbytes,uniform_array_bytes=uniform.nbytes,
                          hash_before_seconds=hash_before_seconds,raw_scan_both_grids_seconds=scan_seconds,
                          hash_after_seconds=hash_after_seconds,largest_chunk_rows=largest_chunk,
                          row_mass_conserved=True,uniform_counted_directly_from_raw=True))
    accumulator=np.zeros((side,side),np.uint64)
    for source in sources:
        accumulator+=source['pilot']
    if accumulator.max(initial=0)>np.iinfo(np.uint32).max:
        raise OverflowError('Global pilot-cell mass exceeds uint32')
    global_pilot=accumulator.astype(np.uint32)
    peak_numeric=max(peak_numeric,persistent_numeric+accumulator.nbytes+global_pilot.nbytes)
    del accumulator
    pd.DataFrame(costs).to_csv(Path(destination)/'raw_construction.csv',index=False)
    state['raw_input_identity']=[dict(source_id=str(r['source_id']),raw_sha256=r['raw_sha256'],
                                      raw_rows=int(r['expected_rows'])) for r in data['raw_manifest']]
    state.setdefault('construction',{}).update(pilot_side=side,uniform_side=uniform_n,
        pilot_total_rows=int(global_pilot.sum(dtype=np.uint64)),persistent_count_input_array_bytes=persistent_numeric+global_pilot.nbytes,
        raw_scan_numeric_array_allocation_bound_bytes=int(peak_numeric),
        memory_measure='Raw-scan numeric-array allocation bound only; excludes pandas/parser/Python objects and later layout-search/summary-index transients; not process peak RSS',
        shared_layout_pilot_is_coordinator_input=True,uniform_counted_directly_from_raw=True,
        raw_semantics='TAB first2 fields; round-trip float parser; every row one point; duplicates retained; trailing fields ignored; internal half-open cells and inclusive outer maxima')
    state.setdefault('timings',{})['raw_construction_with_hashes_seconds']=time.perf_counter()-total_tick
    return sources,global_pilot

## Layout selection receives training geometry, never test geometry or truth

Freeze the selected old training geometry and protocol. Layout selection has no argument containing development geometry, truth or model scores.

In [ ]:
def choose_training_geometry(data, seed, cap):
    outer = data['outer']
    roles = outer[(outer.split_kind == 'cell') & (outer.seed == seed)]
    if roles.empty or roles.qId.duplicated().any():
        raise ValueError(f'Missing or ambiguous original cell split for seed {seed}')
    train = roles[roles.split == 'train'].copy()
    if train.empty:
        raise ValueError(f'No original outer-training geometry for seed {seed}')
    other = roles[roles.split != 'train']
    if set(train.cell) & set(other.cell):
        raise ValueError('Outer cell split leaks generating-cell IDs across training and held-out roles')
    # First cap cells by a fixed hash, then choose one original rectangle/cell by another fixed hash.
    # This prevents repeated jittered queries from dominating the construction workload.
    selected_cells = set(stable_ids(train.cell, cap, '18-training-cell-v1'))
    train = train[train.cell.astype(str).isin(selected_cells)]
    train['selection_hash'] = train.qId.map(lambda q: hashlib.sha256(f'18-training-query-v1:{q}'.encode()).hexdigest())
    train = train.sort_values(['cell', 'selection_hash', 'qId']).drop_duplicates('cell').sort_values('cell')
    ids = train.qId.to_numpy(dtype=int)
    if (ids < 0).any() or (ids >= len(data['old_queries'])).any():
        raise ValueError('Original qId does not index the frozen query geometry')
    rectangles = data['old_queries'][ids, :4].copy()
    selected = train[['seed', 'qId', 'cell']].copy()
    selected[['x0', 'x1', 'y0', 'y1']] = rectangles
    return rectangles, selected


def freeze_protocol(cfg, data, destination, state):
    selected = []; training = {}
    for seed in cfg['seeds']:
        geometry, rows = choose_training_geometry(data, seed, cfg['training_cells'])
        training[seed] = geometry;selected.append(rows)
    pd.concat(selected, ignore_index=True).to_csv(destination/'layout_training_queries.csv', index=False,
                                                float_format='%.17g')
    protocol = dict(version='18-common-layout-bounds-v1', implementation_sha256=IMPLEMENTATION_SHA256,
        methods=list(METHODS), budget_bytes=int(data['budget_bytes']), max_depth=cfg['max_depth'],
        seeds=cfg['seeds'], seeds_role='original outer training-layout replicates, not independent datasets or model training',
        training_cells_cap=cfg['training_cells'], training_geometry_sha256=sha(destination/'layout_training_queries.csv'),
        training_selection='one hash-selected old outer-training rectangle per hash-selected generating cell',
        layout_inputs='exact global dyadic count pilot and ORIGINAL outer-training query geometry only',
        relative_width_objective='greedy reduction of conservative count-interval width; weights inverse coarse64 pilot answer floored at10',
        data_objective='positive leaf mass times 2**(-depth)', uniform='largest square N fitting the SAME actual binary byte cap',
        schema='64-byte header + uint32 exact leaf masses; adaptive adds packed preorder topology; uniform geometry implicit',
        source_coordination='one coordinator chooses a shared immutable template before source compression; arbitrary heterogeneous templates rejected',
        bounds='L=sum fully contained leaf masses; U=sum masses of all geometrically touching leaves; closed rectangles',
        epsilon=EPSILON, certificate='L>0 and 4H<=5L and 4H>=3U, or H=L=U=0',
        zero_truth_error='0 if H=0, otherwise infinity', runtime_lower_zero='no relative certificate when L=0<U; no min_exact substitution',
        quality_cohort='frozen17 eligible truth>=10, only offline; all candidates including zeros audited for validity',
        reference='same inspected17 queries and exact raw labels; DEVELOPMENT, not a new untouched validation',
        model_training=False, acceptance_calibration=False, test_based_layout_selection=False,
        production_e2e=False, query_cache='resident decoded summary + spatial directory, no pilot/raw/truth lookup',
        source17_manifest_sha256=state['source17_manifest_sha256'])
    save_json(destination/'protocol_lock.json', protocol)
    state['protocol_sha256'] = sha(destination/'protocol_lock.json');state['protocol'] = protocol
    save_json(destination/'run.json', state)
    return training

## Persist source summaries, then merge only those immutable count artifacts

Construct each layout, serialize source summaries, reload and add exact leaf counts. Compare one-shot, sequential, balanced and reverse merges; reload the final artifact for runtime querying.

In [ ]:
def validate_and_persist_sources(method, seed, source_counts, layout, global_pilot, data, destination, costs):
    directory = destination/'summaries'/f'{method}-{seed}'
    directory.mkdir(parents=True, exist_ok=False)
    paths = [];construction_start = time.perf_counter()
    for record in source_counts:
        summary = (uniform_summary(record['uniform'], data['root']) if method == 'uniform'
                   else source_summary(record['pilot'], layout))
        path = directory/f"source_{record['source_id']}.summary"
        written = save_summary(path, summary)
        if written > data['budget_bytes']:
            raise ValueError('A source summary exceeds its declared budget')
        paths.append(path)
    costs.append(dict(method=method, seed=seed, stage='source_count_projection_and_serialization',
                      seconds=time.perf_counter()-construction_start))
    # Reconstruct an independent single-union reference BEFORE entering the summary-only merge path.
    if method == 'uniform':
        counts = np.sum([record['uniform'].astype(np.uint64) for record in source_counts], axis=0, dtype=np.uint64)
        one_shot = uniform_summary(counts, data['root'])
    else:
        one_shot = source_summary(global_pilot, layout)
    expected = one_shot.counts.copy();expected_total = one_shot.total_count
    del summary, one_shot
    load_start = time.perf_counter();sources = [load_summary(path) for path in paths]
    costs.append(dict(method=method, seed=seed, stage='source_artifact_load_and_index', seconds=time.perf_counter()-load_start))
    # No raw path, pilot, query or label is an argument of merge_summaries.
    merge_start = time.perf_counter();merged = merge_summaries(sources, data['budget_bytes'])
    merge_seconds = time.perf_counter()-merge_start
    if not np.array_equal(expected, merged.counts):
        raise ValueError('Merged counts differ from exact single-union construction')
    audit_start = time.perf_counter();sequential = sources[0]
    for source in sources[1:]:
        sequential = merge_summaries([sequential, source], data['budget_bytes'])
    pending = list(sources)
    while len(pending) > 1:
        pending = [merge_summaries(pending[i:i+2], data['budget_bytes']) for i in range(0, len(pending), 2)]
    balanced = pending[0];reverse = merge_summaries(reversed(sources), data['budget_bytes'])
    for candidate in (sequential, balanced, reverse):
        if candidate.layout_hash != merged.layout_hash or not np.array_equal(candidate.counts, expected):
            raise ValueError('A summary-only merge order changed exact leaf counts')
    costs.append(dict(method=method, seed=seed, stage='audit_only_merge_orders', seconds=time.perf_counter()-audit_start))
    # Save and reload the proposed runtime artifact, so no construction-only buffers can enter querying.
    merged_path = directory/'merged.summary';tick = time.perf_counter();nbytes = save_summary(merged_path, merged)
    costs.append(dict(method=method, seed=seed, stage='merged_artifact_serialize', seconds=time.perf_counter()-tick,
                      serialized_bytes=nbytes))
    costs.append(dict(method=method, seed=seed, stage='summary_only_merge_with_index', seconds=merge_seconds,
                      serialized_bytes=nbytes))
    del sources, sequential, pending, balanced, reverse, merged
    tick = time.perf_counter();resident = load_summary(merged_path);load_seconds = time.perf_counter()-tick
    costs.append(dict(method=method, seed=seed, stage='merged_artifact_read_decode_and_index', seconds=load_seconds,
                      serialized_bytes=nbytes))
    audit = dict(method=method, seed=seed, source_count=len(paths), expected_total=expected_total,
                 merged_total=resident.total_count, exact_oneshot=True, exact_sequential=True, exact_balanced=True,
                 exact_reverse=True, within_budget=nbytes<=data['budget_bytes'], serialized_bytes=nbytes,
                 source_collection_bytes=sum(path.stat().st_size for path in paths),
                 layout_hash=resident.layout_hash, leaf_count=resident.leaf_count,
                 derived_numpy_bytes=resident.derived_nbytes,
                 directory_construction_named_nbytes=resident.directory_construction_named_nbytes,
                 resident_numpy_bytes=resident.derived_nbytes+resident.counts.nbytes+resident.topology.nbytes,
                 artifact_sha256=sha(merged_path), artifact_path=str(merged_path))
    return resident, audit


def build_candidates(cfg, data, training, destination, state):
    construction_start = time.perf_counter()
    budget = int(data['budget_bytes']);uniform_n = int(math.isqrt((budget-HEADER.size)//4))
    if uniform_n < 1 or HEADER.size+4*(uniform_n+1)**2 <= budget:
        raise ValueError('Uniform budget calculation is invalid')
    tick = time.perf_counter()
    records, global_pilot = construct_count_inputs(data, cfg['max_depth'], uniform_n, destination, state)
    state['timings']['raw_counts_construction_seconds'] = time.perf_counter()-tick
    state.update(uniform_n=uniform_n, uniform_serialized_bytes=HEADER.size+4*uniform_n**2)
    candidates = {};audits = [];layouts = [];costs = []
    # Uniform and data-only layout are seed-invariant controls, constructed once and referenced by all seeds.
    for method in METHODS:
        seeds = cfg['seeds'] if method == 'adaptive_relative_width' else [-1]
        for seed in seeds:
            tick = time.perf_counter()
            layout = (None if method == 'uniform' else build_layout(global_pilot, data['root'],
                training[seed] if method == 'adaptive_relative_width' else None, budget,
                strategy='relative_width' if method == 'adaptive_relative_width' else 'data', max_depth=cfg['max_depth']))
            selection_seconds = time.perf_counter()-tick
            costs.append(dict(method=method, seed=seed, stage='layout_selection', seconds=selection_seconds))
            print(f'Constructing {method}, layout seed {seed}; layout selection {selection_seconds:.3f}s', flush=True)
            resident, audit = validate_and_persist_sources(method, seed, records, layout, global_pilot, data, destination, costs)
            if method == 'adaptive_relative_width':
                candidates[(method, seed)] = resident
            else:
                for target_seed in cfg['seeds']:
                    candidates[(method, target_seed)] = resident
            audits.append(audit)
            layouts.append(dict(method=method, seed=seed, leaf_count=resident.leaf_count,
                layout_hash=resident.layout_hash, actual_max_depth=resident.max_depth, serialized_bytes=resident.serialized_nbytes,
                unused_budget_bytes=budget-resident.serialized_nbytes, uniform_n=uniform_n if method=='uniform' else 0,
                training_cells=len(training[seed]) if method=='adaptive_relative_width' else 0,
                pilot_pyramid_nbytes=layout['pilot_pyramid_nbytes'] if layout else 0,
                layout_search_memory_scope=layout['layout_search_memory_scope'] if layout else 'No adaptive layout search',
                construction_seconds=selection_seconds))
            pd.DataFrame(costs).to_csv(destination/'costs.csv', index=False)
            pd.DataFrame(audits).to_csv(destination/'merge_audit.csv', index=False)
            pd.DataFrame(layouts).to_csv(destination/'layouts.csv', index=False)
    # Deliberately release exact pilots/direct source grids before ANY candidate evaluation.
    del records, global_pilot, layout
    state['construction_arrays_released_before_query'] = True
    state['timings']['all_candidate_build_seconds'] = time.perf_counter()-construction_start
    return candidates

## All-candidate interval correctness; positive-count utility and resident query costs

Audit all candidate geometries before reporting utility on truth≥10. Compare the same queries and randomized method-call order, warm up each method, and keep truth checks outside the query timer.

In [ ]:
def actual_relative_error(estimate, truth):
    if truth == 0:
        return 0. if estimate == 0 else np.inf
    return abs(estimate-truth)/truth


def evaluate_candidates(cfg, data, candidates, destination, state):
    labels = data['labels'].set_index('query_id', verify_integrity=True)
    membership = data['membership'][data['membership'].seed.isin(cfg['seeds'])]
    cache = {};rows = [];all_query_audit = []
    # Same geometry is evaluated once per distinct layout. Repeated seed memberships remain explicitly identified.
    for (method, seed), summary in candidates.items():
        if summary.layout_hash not in cache:
            values = {}
            for query_id, row in labels.iterrows():
                q = row[['x0', 'x1', 'y0', 'y1']].to_numpy(float)
                values[query_id] = query_summary(summary, q)
                result = values[query_id];truth = int(row.truth)
                all_query_audit.append(dict(layout_hash=summary.layout_hash, method=method, layout_seed=seed,
                    query_id=query_id, family=row.family, truth=truth,
                    interval_valid=result['lower']<=truth<=result['upper'], certified=result['certified'],
                    certificate_violation=result['certified'] and actual_relative_error(result['estimate'], truth)>EPSILON))
            cache[summary.layout_hash] = values
        values = cache[summary.layout_hash]
        member = membership[membership.seed == seed]
        for item in member.itertuples(index=False):
            label = labels.loc[item.query_id];result = values[item.query_id]
            truth = int(label.truth);error = actual_relative_error(result['estimate'], truth)
            rows.append(dict(method=method, seed=seed, family=item.family, cell=item.cell, query_id=item.query_id,
                truth=truth, eligible=truth>=MIN_EXACT, zero_truth=truth==0, actual_relative_error=error,
                within25=error<=EPSILON, interval_valid=result['lower']<=truth<=result['upper'],
                **result))
    frame = pd.DataFrame(rows)
    frame.to_csv(destination/'per_query.csv', index=False, float_format='%.17g')
    all_query_audit = pd.DataFrame(all_query_audit)
    all_query_audit.to_csv(destination/'interval_audit_all_queries.csv', index=False)
    audits = pd.read_csv(destination/'merge_audit.csv');metrics = [];validity = []
    for (family, seed, method), all_part in frame.groupby(['family', 'seed', 'method'], sort=True):
        certified_all = all_part[all_part.certified]
        validity.append(dict(family=family, seed=seed, method=method, n_candidates=len(all_part),
            interval_failures=int((~all_part.interval_valid).sum()), certified_count=int(all_part.certified.sum()),
            certified_violations=int((certified_all.actual_relative_error>EPSILON).sum()),
            zero_truth_count=int(all_part.zero_truth.sum()), small_positive_count=int(((all_part.truth>0)&(all_part.truth<MIN_EXACT)).sum())))
        part = all_part[all_part.eligible];n = len(part);accepted = part[part.certified]
        audit = audits[(audits.method==method)&audits.seed.isin([seed, -1])].iloc[0]
        metrics.append(dict(family=family, seed=seed, method=method, n_queries=n,
            n_unique_queries=part.query_id.nunique(), n_unique_cells=part.cell.nunique(),
            within25_count=int(part.within25.sum()), within25_fraction=float(part.within25.mean()) if n else np.nan,
            actual_error_mean=float(part.actual_relative_error.mean()) if n else np.nan,
            actual_error_median=float(part.actual_relative_error.median()) if n else np.nan,
            actual_error_p95=float(part.actual_relative_error.quantile(.95)) if n else np.nan,
            certified_count=len(accepted), certified_fraction=len(accepted)/n if n else np.nan,
            certified_violations=int((accepted.actual_relative_error>EPSILON).sum()),
            certified_violation_rate=float((accepted.actual_relative_error>EPSILON).mean()) if len(accepted) else np.nan,
            good_answer_recall=float(accepted.within25.sum()/part.within25.sum()) if part.within25.sum() else np.nan,
            bound_failures=int((~part.interval_valid).sum()), lower_zero_count=int((part.lower==0).sum()),
            interval_width_median=float(part.interval_width.median()) if n else np.nan,
            interval_width_p90=float(part.interval_width.quantile(.9)) if n else np.nan,
            relative_interval_width_median=float((part.interval_width/part.truth.clip(lower=1)).median()) if n else np.nan,
            serialized_bytes=int(audit.serialized_bytes), derived_numpy_bytes=int(audit.derived_numpy_bytes),
            resident_numpy_bytes=int(audit.resident_numpy_bytes)))
    validity = pd.DataFrame(validity);validity.to_csv(destination/'validity.csv', index=False)
    pd.DataFrame(metrics).to_csv(destination/'metrics.csv', index=False)
    state['candidate_query_cases'] = len(frame)
    state['unique_audited_queries'] = len(labels)
    state['all_interval_failures'] = int((~all_query_audit.interval_valid).sum())
    state['all_certificate_violations'] = int(all_query_audit.certificate_violation.sum())
    if state['all_interval_failures'] or state['all_certificate_violations']:
        raise ValueError('Correctness blocker: interval containment or certified answer violated; inspect validity.csv')
    # Hash-selected common positive-count queries, identical IDs/order for every method within each seed/family.
    times = []; timing_ids = []
    for (family, seed), members in membership.groupby(['family', 'seed'], sort=True):
        eligible = members[members.query_id.isin(labels[labels.truth>=MIN_EXACT].index)]
        ids = stable_ids(eligible.query_id, cfg['timing_queries'], '18-common-resident-timing-v1')
        if not ids:
            continue
        for method in METHODS:
            for query_id in ids[:min(5, len(ids))]:
                query_summary(candidates[(method, seed)], labels.loc[query_id, ['x0','x1','y0','y1']].to_numpy(float))
        timing_ids.extend(dict(family=family, seed=seed, query_id=qid) for qid in ids)
        for repeat in range(cfg['timing_repeats']):
            calls = [(method, query_id) for query_id in ids for method in METHODS]
            calls.sort(key=lambda pair: hashlib.sha256(f'18-call-order-v1:{seed}:{family}:{repeat}:{pair}'.encode()).digest())
            for method, query_id in calls:
                summary = candidates[(method, seed)]
                q = labels.loc[query_id, ['x0','x1','y0','y1']].to_numpy(float)
                tick = time.perf_counter();result = query_summary(summary, q);seconds = time.perf_counter()-tick
                expected = cache[summary.layout_hash][query_id]
                if result != expected:
                    raise ValueError('Resident timed output changed from untimed summary-only query')
                times.append(dict(family=family, seed=seed, method=method, query_id=query_id, repeat=repeat,
                                  seconds=seconds, resident_cache=True))
    timing = pd.DataFrame(times, columns=['family','seed','method','query_id','repeat','seconds','resident_cache'])
    timing.to_csv(destination/'query_times.csv', index=False)
    pd.DataFrame(timing_ids, columns=['family','seed','query_id']).to_csv(destination/'timing_query_ids.csv', index=False)
    summaries = []
    for keys, group in timing.groupby(['family','seed','method']):
        summaries.append(dict(zip(['family','seed','method'], keys), n_calls=len(group),
            n_unique_queries=group.query_id.nunique(), repeats=cfg['timing_repeats'],
            mean_seconds=float(group.seconds.mean()), p95_seconds=float(group.seconds.quantile(.95)),
            cache='resident decoded summary and derived spatial directory; no raw/pilot/truth read'))
    pd.DataFrame(summaries).to_csv(destination/'timing_summary.csv', index=False)
    return frame

## English report and three research-question figures

Reporting reads saved measurements only. It cannot select a layout or change an acceptance decision.

In [ ]:
"""English descriptive reports for the fixed-budget adaptive-bound experiment.

Reporting consumes saved measurements only. It never selects a layout, changes
an acceptance rule, reads raw points, or fits a model. Seed summaries are
descriptive; the same query appears under multiple layout seeds.
"""
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd


REPORT_METHODS = ('uniform', 'adaptive_data', 'adaptive_relative_width')
REPORT_LABELS = {
    'uniform': 'Uniform finer',
    'adaptive_data': 'Adaptive: data only',
    'adaptive_relative_width': 'Adaptive: training geometry',
}
REPORT_COLORS = {'uniform': '#31688e', 'adaptive_data': '#35b779',
                 'adaptive_relative_width': '#b65d24'}


def _report_read(path):
    path = Path(path)
    if not path.is_file() or path.stat().st_size == 0:
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()


def _report_number(value, digits=3, percent=False):
    try:
        value = float(value)
    except (TypeError, ValueError):
        return 'undefined'
    if math.isnan(value):
        return 'undefined'
    if math.isinf(value):
        return 'infinite' if value > 0 else '-infinite'
    if percent:
        return f'{100 * value:.{digits}f}%'
    if value.is_integer() and digits == 0:
        return str(int(value))
    return f'{value:.{digits}f}'


def _report_table(frame, specifications):
    """Render a small dependency-free Markdown table.

    specifications maps displayed column name to (source column, formatter).
    """
    if frame.empty:
        return 'No measurements are available.\n'
    header = list(specifications)
    rows = ['| ' + ' | '.join(header) + ' |',
            '| ' + ' | '.join('---' for _ in header) + ' |']
    for _, row in frame.iterrows():
        values = []
        for source, formatter in specifications.values():
            value = row.get(source, np.nan)
            rendered = formatter(value) if formatter else str(value)
            values.append(rendered.replace('|', '/').replace('\n', ' '))
        rows.append('| ' + ' | '.join(values) + ' |')
    return '\n'.join(rows) + '\n'


def _report_families(frame):
    if frame.empty or 'family' not in frame:
        return []
    available = set(frame.family.dropna().astype(str))
    return [s for s in ('new_groups', 'existing_test_groups') if s in available] + sorted(
        available - {'new_groups', 'existing_test_groups'})


def _report_family_label(family):
    return {'new_groups': 'New generating groups (primary development cohort)',
            'existing_test_groups': 'Existing held-out groups (supplement)'}.get(family, family)


def _report_summary_timing(times):
    if times.empty or not {'family', 'seed', 'method', 'seconds'}.issubset(times.columns):
        return pd.DataFrame(columns=['family', 'seed', 'method', 'calls', 'mean_ms', 'p95_ms'])
    rows = []
    for (family, seed, method), part in times.groupby(['family', 'seed', 'method'], dropna=False):
        values = part.seconds.to_numpy(float) * 1000
        rows.append(dict(family=family, seed=seed, method=method, calls=len(part),
                         mean_ms=float(np.mean(values)), p95_ms=float(np.quantile(values, .95))))
    return pd.DataFrame(rows)


def _report_paired(metrics, timing):
    """Exact matched cohort/seed comparisons; no independent-seed significance."""
    rows = []
    if metrics.empty:
        return pd.DataFrame()
    timing_lookup = {} if timing.empty else timing.set_index(['family', 'seed', 'method']).mean_ms.to_dict()
    for (family, seed), part in metrics.groupby(['family', 'seed'], dropna=False):
        base = part[part.method == 'uniform']
        if len(base) != 1:
            continue
        base = base.iloc[0]
        for method in ('adaptive_data', 'adaptive_relative_width'):
            candidate = part[part.method == method]
            if len(candidate) != 1:
                continue
            candidate = candidate.iloc[0]
            baseline_time = timing_lookup.get((family, seed, 'uniform'), np.nan)
            candidate_time = timing_lookup.get((family, seed, method), np.nan)
            rows.append(dict(family=family, seed=seed, method=method,
                delta_certified_fraction=float(candidate.certified_fraction-base.certified_fraction),
                delta_within25_fraction=float(candidate.within25_fraction-base.within25_fraction),
                delta_relative_width=float(candidate.relative_interval_width_median-base.relative_interval_width_median),
                resident_mean_time_ratio=float(candidate_time/baseline_time) if baseline_time > 0 else np.nan,
                candidate_bytes=int(candidate.serialized_bytes), uniform_bytes=int(base.serialized_bytes),
                certified_violations=int(candidate.certified_violations),
                bound_failures=int(candidate.bound_failures)))
    return pd.DataFrame(rows)


def _report_recommendation(metrics, paired, validity):
    if not validity.empty and (validity.interval_failures.sum() > 0 or validity.certified_violations.sum() > 0):
        return ('A correctness check failed. Stop all quality/certification claims and fix the implementation before '
                'running further performance experiments. Neither a narrow interval nor a higher acceptance fraction '
                'is useful if the count interval excludes the true answer.'), 'correctness_blocker'
    if paired.empty:
        return ('Matched uniform comparisons are missing. Complete those measurements before judging this candidate.'), 'incomplete'
    primary = paired[paired.family == 'new_groups']
    if primary.empty:
        primary = paired
    viable = []
    for method, part in primary.groupby('method'):
        # Both point quality and certificate coverage are shown. This deliberately
        # does not turn a post-hoc gain into an inferential success claim.
        coverage_consistent = bool((part.delta_certified_fraction >= -1e-12).all() and
                                   (part.delta_certified_fraction > 1e-12).any())
        quality_consistent = bool((part.delta_within25_fraction >= -1e-12).all() and
                                  (part.delta_within25_fraction > 1e-12).any())
        if coverage_consistent or quality_consistent:
            viable.append(REPORT_LABELS.get(method, method))
    any_positive = bool(((primary.delta_certified_fraction > 1e-12) |
                         (primary.delta_within25_fraction > 1e-12)).any())
    if viable:
        return (f"A consistent descriptive gain across the saved primary layout-seed comparisons appears for: {', '.join(viable)}. "
                'Inspect the paired coverage, point-quality and latency tradeoffs below. If the gain is useful at the '
                'reported construction/storage/runtime costs, freeze exactly one candidate and its protocol, then validate '
                'it on untouched queries or another dataset. These inspected queries cannot provide the final validation '
                'or establish statistical significance. Do not change the split priority on these labels.'), 'development_gain'
    if not any_positive:
        return ('Neither adaptive variant improves certified coverage or the within-25% point-answer fraction on any '
                'matched primary layout-seed comparison. Stop this candidate under this budget/workload rather than '
                'searching split priorities on the inspected labels. Uniform finer remains the stronger control for '
                'these measured outcomes; this result does not prove that all adaptive summaries are inferior.'), 'stop_no_measured_gain'
    return ('The primary comparisons are mixed across layout seeds or outcomes. There is no clear advantage yet. '
            'Report the tradeoff and do not promote it as an effective replacement or tune it on these test labels. '
            'A follow-up is justified only by a specific independent mechanism or deployment requirement.'), 'mixed_development_evidence'


def _report_figures(destination, metrics, timing):
    import matplotlib.pyplot as plt
    plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False,
                         'figure.dpi': 130})
    families = _report_families(metrics)
    if not families:
        return []
    methods = [m for m in REPORT_METHODS if m in set(metrics.method)]
    colors = [REPORT_COLORS[m] for m in methods]
    names = [REPORT_LABELS[m] for m in methods]
    x = np.arange(len(methods))

    def draw(ax, part, column, scale=1):
        values = []
        for i, method in enumerate(methods):
            a = part.loc[part.method == method, column].to_numpy(float)*scale
            values.append(float(np.nanmean(a)) if np.isfinite(a).any() else np.nan)
            if len(a):
                offset = np.linspace(-.09, .09, len(a)) if len(a) > 1 else np.array([0.])
                ax.scatter(i+offset, a, color=REPORT_COLORS[method], edgecolor='white', linewidth=.4, s=28, zorder=3)
        ax.bar(x, values, color=colors, alpha=.28, width=.55)
        ax.set_xticks(x, names, rotation=17, ha='right')
        ax.grid(axis='y', color='#dddddd', linewidth=.6, alpha=.7)

    fig, axes = plt.subplots(1, len(families), figsize=(6.2*len(families), 4.3), squeeze=False)
    for index, family in enumerate(families):
        ax = axes[0, index]
        draw(ax, metrics[metrics.family == family], 'certified_fraction', 100)
        ax.set_title(_report_family_label(family), fontsize=11)
        ax.set_ylabel('Certified within 25% of truth (%)')
        peak = float(metrics.loc[metrics.family == family, 'certified_fraction'].max())*100
        ax.set_ylim(0, max(1., peak*1.3))
        for i, method in enumerate(methods):
            value = metrics.loc[(metrics.family == family)&(metrics.method == method), 'certified_fraction'].mean()*100
            ax.text(i, value+max(.025, peak*.04), f'{value:.2f}%', ha='center', fontsize=9)
    fig.suptitle('How many positive-count answers can be certified?\nBars: descriptive seed mean; dots: saved layout seeds (not independent trials)', fontsize=12)
    fig.tight_layout(rect=(0, 0, 1, .89))
    p = destination/'01_certified_coverage.png';fig.savefig(p, bbox_inches='tight');plt.close(fig)

    fig, axes = plt.subplots(len(families), 2, figsize=(12, 4.1*len(families)), squeeze=False)
    for row, family in enumerate(families):
        part = metrics[metrics.family == family]
        draw(axes[row, 0], part, 'within25_fraction', 100)
        axes[row, 0].set_title(_report_family_label(family), fontsize=11)
        axes[row, 0].set_ylabel('All point answers within 25% (%)')
        axes[row, 0].set_ylim(0, 102)
        draw(axes[row, 1], part, 'relative_interval_width_median')
        axes[row, 1].set_title('Conservative interval tightness (lower is better)', fontsize=11)
        axes[row, 1].set_ylabel('Median (upper − lower) / truth')
        axes[row, 1].set_ylim(bottom=0)
    fig.suptitle('Answer quality and interval width answer different questions\nThe three layouts change the answer itself; each uses the same certificate rule', fontsize=12)
    fig.tight_layout(rect=(0, 0, 1, .93))
    p2 = destination/'02_quality_and_width.png';fig.savefig(p2, bbox_inches='tight');plt.close(fig)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
    distinct = metrics.drop_duplicates(['seed', 'method'])
    draw(axes[0], distinct, 'serialized_bytes', 1/1024)
    axes[0].set_ylabel('Serialized summary (KiB)');axes[0].set_title('Actual binary bytes; no model')
    draw(axes[1], distinct, 'derived_numpy_bytes', 1/(1024**2))
    axes[1].set_ylabel('Derived numeric arrays (MiB)');axes[1].set_title('Loaded index/bounds overhead')
    if timing.empty:
        axes[2].text(.5, .5, 'No timing measurements', ha='center', va='center', transform=axes[2].transAxes)
        axes[2].set_axis_off()
    else:
        timing_part = timing[timing.family == ('new_groups' if 'new_groups' in set(timing.family) else timing.family.iloc[0])]
        draw(axes[2], timing_part, 'mean_ms')
        axes[2].set_ylabel('Mean resident query time (ms)')
        axes[2].set_title('Estimate + interval + certificate')
    fig.suptitle('Cost: serialized bytes are not process RAM; resident query time is not end-to-end latency', fontsize=12)
    fig.tight_layout(rect=(0, 0, 1, .91))
    p3 = destination/'03_storage_and_resident_cost.png';fig.savefig(p3, bbox_inches='tight');plt.close(fig)
    return [p.name, p2.name, p3.name]


def write_reports(destination, state):
    """Write the English report and three figures, without mutating decisions."""
    destination = Path(destination)
    metrics = _report_read(destination/'metrics.csv')
    times = _report_read(destination/'query_times.csv')
    costs = _report_read(destination/'costs.csv')
    validity = _report_read(destination/'validity.csv')
    merge = _report_read(destination/'merge_audit.csv')
    raw_costs = _report_read(destination/'raw_construction.csv')
    timing = _report_summary_timing(times)
    timing.to_csv(destination/'query_timing_summary.csv', index=False)
    paired = _report_paired(metrics, timing)
    paired.to_csv(destination/'paired_vs_uniform.csv', index=False)
    recommendation, status = _report_recommendation(metrics, paired, validity)
    figures = _report_figures(destination, metrics, timing)
    cfg = state.get('config', state.get('cfg', {}))
    budget = state.get('budget_bytes', cfg.get('budget_bytes', 'inherited from the source raw2 summary budget'))
    report = [
        '# 18: Budgeted adaptive counts with conservative query bounds',
        '',
        '**Status:** ' + status.replace('_', ' ') + '.',
        '',
        'This experiment asks whether coordinated adaptive space allocation improves useful count-error certification '
        'or point-answer quality over a compact uniform finer histogram at the same serialized-byte cap. '
        'It does not train a model, recalibrate a threshold, or certify the older estimated ARM bucket masses.',
        '',
        '## Scope and fixed comparison',
        '',
        '- The queries and exact labels from notebook 17 have already been inspected. They are **development evidence**, '
        'not a new final validation set. The primary new-generating-group cohort and supplementary existing-held-out-group '
        'cohort remain separate. They use the same raw dataset, not a new geographic dataset.',
        '- Every method uses exact original point-row counts in its own partition. Uniform counts are obtained directly '
        'from raw points, never interpolated from the dyadic pilot. Different layouts change the approximate answer itself.',
        '- The adaptive partition is selected once with a coordinator before source compression. Every source uses exactly '
        'that shared layout. Merging reloads persisted source summaries and adds matching leaf counts. Different partitions '
        'are rejected; arbitrary heterogeneous ARM merging has not been solved.',
        '- Data-only adaptation uses point mass. Workload adaptation uses only saved outer-training query geometry and '
        'coarse-pilot count estimates for weights. No development-query labels or geometry select its layout. Seeds identify '
        'saved training splits/layouts, not independent repetitions of the same test queries.',
        f'- The actual cap is **{budget} bytes per merged summary**, including the binary header, packed topology (adaptive) '
        'and integer counts. Uniform receives the same representation advantage: implicit boundaries, compact integer '
        'counts and the same certificate engine. No model file is required.',
        '',
        '## What the certificate means',
        '',
        'A closed query receives a lower count L from fully enclosed leaves and an upper count U from every intersecting '
        'leaf, including boundary-only contact. Because these leaf masses are exact, L ≤ true count ≤ U. '
        'The area-interpolated answer H is certified only if every possible true count in this interval gives relative '
        'error at most 25%. For L > 0, the rule is H ≤ 1.25L and H ≥ 0.75U, checked using exact integer-ratio comparisons. '
        'If L = 0 < U, no certificate is issued; the evaluation floor of 10 is never substituted for L. '
        'An exact empty interval with H = L = U = 0 is certified. At true count zero, actual relative error is 0 if H = 0 '
        'and infinite otherwise.',
        '',
        'Interval containment and certificate safety are audited on **all generated candidates**, including zero and '
        'small positive counts. The utility tables use the inherited positive-count cohort true count ≥ 10. '
        'A zero certified count has an **undefined** certified violation rate, not 0%. A valid but wide interval can be '
        'safe and still practically unhelpful.',
        '',
        '## All-candidate correctness audit',
        '',
        _report_table(validity, {
            'Cohort': ('family', None), 'Seed': ('seed', lambda x: _report_number(x, 0)),
            'Method': ('method', lambda x: REPORT_LABELS.get(x, x)),
            'Candidates': ('n_candidates', lambda x: _report_number(x, 0)),
            'Interval failures': ('interval_failures', lambda x: _report_number(x, 0)),
            'Certified': ('certified_count', lambda x: _report_number(x, 0)),
            'Certified violations': ('certified_violations', lambda x: _report_number(x, 0)),
            'Zero truth': ('zero_truth_count', lambda x: _report_number(x, 0)),
        }),
    ]
    for family in _report_families(metrics):
        part = metrics[metrics.family == family]
        report += ['', '## ' + _report_family_label(family), '',
            'Each row is one saved layout seed on the same cohort. Do not pool repeated seed/query cases as independent '
            'observations or infer significance from these descriptive differences.', '',
            _report_table(part, {
                'Seed': ('seed', lambda x: _report_number(x, 0)),
                'Method': ('method', lambda x: REPORT_LABELS.get(x, x)),
                'Queries': ('n_queries', lambda x: _report_number(x, 0)),
                'All answers within 25%': ('within25_fraction', lambda x: _report_number(x, 2, True)),
                'Certified': ('certified_count', lambda x: _report_number(x, 0)),
                'Certified fraction': ('certified_fraction', lambda x: _report_number(x, 2, True)),
                'Certified violations': ('certified_violations', lambda x: _report_number(x, 0)),
                'Certified violation rate': ('certified_violation_rate', lambda x: _report_number(x, 2, True)),
                'Median interval / truth': ('relative_interval_width_median', lambda x: _report_number(x, 3)),
            }),
            '', 'All-answer quality uses each layout’s own point answer. It is not a comparison of ML prediction MAE '
            'against different target errors. A better point answer and a more informative quality certificate are '
            'distinct outcomes. The interval-width diagnostic uses truth offline; it is not a query-time feature.',
            '',
            _report_table(part, {
                'Seed': ('seed', lambda x: _report_number(x, 0)),
                'Method': ('method', lambda x: REPORT_LABELS.get(x, x)),
                'Mean actual error': ('actual_error_mean', lambda x: _report_number(x, 2, True)),
                'Median actual error': ('actual_error_median', lambda x: _report_number(x, 2, True)),
                'P95 actual error': ('actual_error_p95', lambda x: _report_number(x, 2, True)),
                'Median interval width': ('interval_width_median', lambda x: _report_number(x, 1)),
                'P90 interval width': ('interval_width_p90', lambda x: _report_number(x, 1)),
                'L = 0': ('lower_zero_count', lambda x: _report_number(x, 0)),
            })]
    report += ['', '## Matched differences from uniform', '',
        'Positive certified-fraction or within-25%-fraction deltas favor adaptive. A negative interval-width delta favors '
        'adaptive. A resident-time ratio above 1 means adaptive is slower. These are paired descriptive comparisons, '
        'not independent-trial confidence intervals.', '',
        _report_table(paired, {
            'Cohort': ('family', None), 'Seed': ('seed', lambda x: _report_number(x, 0)),
            'Method': ('method', lambda x: REPORT_LABELS.get(x, x)),
            'Certified delta (pp)': ('delta_certified_fraction', lambda x: _report_number(100*x, 2)),
            'Within-25% delta (pp)': ('delta_within25_fraction', lambda x: _report_number(100*x, 2)),
            'Median width/truth delta': ('delta_relative_width', lambda x: _report_number(x, 3)),
            'Resident mean time ratio': ('resident_mean_time_ratio', lambda x: _report_number(x, 2)),
        }),
        '', '## Merge and cost accounting', '',
        _report_table(merge, {
            'Seed': ('seed', lambda x: _report_number(x, 0)),
            'Method': ('method', lambda x: REPORT_LABELS.get(x, x)),
            'Sources': ('source_count', lambda x: _report_number(x, 0)),
            'Expected rows': ('expected_total', lambda x: _report_number(x, 0)),
            'Merged rows': ('merged_total', lambda x: _report_number(x, 0)),
            'One-shot exact': ('exact_oneshot', None), 'Sequential exact': ('exact_sequential', None),
            'Balanced exact': ('exact_balanced', None), 'Reverse exact': ('exact_reverse', None),
            'Within cap': ('within_budget', None),
        }),
        '',
        _report_table(merge, {
            'Layout seed': ('seed', lambda x: _report_number(x, 0)),
            'Method': ('method', lambda x: REPORT_LABELS.get(x, x)),
            'Merged bytes': ('serialized_bytes', lambda x: _report_number(x, 0)),
            'All source files (bytes)': ('source_collection_bytes', lambda x: _report_number(x, 0)),
            'Derived index/bounds bytes': ('derived_numpy_bytes', lambda x: _report_number(x, 0)),
            'Retained numeric bytes': ('resident_numpy_bytes', lambda x: _report_number(x, 0)),
        }),
        '',
        'Layout seed -1 denotes a single seed-invariant control reused across the saved memberships. Numeric memory '
        'includes counts, unpacked topology, bounds and directory arrays; it remains a lower bound on process memory.',
        '',
        'The raw pass, temporary pilot atlas, shared-layout search, source compression, input loading, count merge and '
        'output serialization are construction or initialization costs and are recorded separately. Each source '
        'temporarily pays for an atlas during coordinated construction; this is not a one-pass independent-source ARM '
        'merge protocol. The merged summary does not retain pilot counts, raw points or training queries.',
        '',
        'Serialized bytes are **not process RAM**. The loaded numeric leaf bounds and spatial directory have additional '
        'reported memory; Python object overhead and transient query arrays require more memory still. No model-loading '
        'cost is hidden because no model is used. The persisted per-source summaries also occupy storage during ingestion '
        'and are not included in the cap for the final merged artifact.',
        '',
        _report_table(raw_costs, {
            'Source': ('source_id', None),
            'Raw rows': ('raw_rows', lambda x: _report_number(x, 0)),
            'Read + build both grids (s)': ('raw_scan_both_grids_seconds', lambda x: _report_number(x, 4)),
            'Hash before (s)': ('hash_before_seconds', lambda x: _report_number(x, 4)),
            'Hash after (s)': ('hash_after_seconds', lambda x: _report_number(x, 4)),
        }),
        '',
        'Shared raw construction creates both exact grids in one pass for this comparison. It is not a standalone '
        'construction-speed comparison of the methods. Temporary numeric allocation diagnostics are in run.json; '
        'they exclude parser/Python objects and are not measured peak RSS.',
        '',
        'Construction allocation diagnostics (bytes, not process RSS): `' +
        json.dumps(state.get('construction', {}), sort_keys=True) + '`.',
        '',
        'Resident query timing includes point estimation, bound computation and the 25% decision. It excludes artifact '
        'loading, raw ingestion, construction and exact fallback. It is **not an end-to-end query speedup measurement**. '
        'The timings use the same saved query IDs, repetitions, warmups and resident cache setting across methods. '
        'Previous notebook 09 speedups cannot be attributed to this candidate.', '',
        _report_table(timing, {
            'Cohort': ('family', None), 'Seed': ('seed', lambda x: _report_number(x, 0)),
            'Method': ('method', lambda x: REPORT_LABELS.get(x, x)),
            'Calls': ('calls', lambda x: _report_number(x, 0)),
            'Mean resident ms': ('mean_ms', lambda x: _report_number(x, 3)),
            'P95 resident ms': ('p95_ms', lambda x: _report_number(x, 3)),
        }),
        '',
        _report_table(costs, {
            'Seed': ('seed', lambda x: _report_number(x, 0)),
            'Method': ('method', lambda x: REPORT_LABELS.get(x, x)),
            'Stage': ('stage', None), 'Seconds': ('seconds', lambda x: _report_number(x, 4)),
        }),
        '', '## What the evidence supports next', '', recommendation, '',
        'Adaptive partitioning, quadtrees, workload adaptation, count bounds and shared-template addition all have prior '
        'art. Implementing them alone does not establish novelty. A defensible contribution needs a useful demonstrated '
        'budget/quality/cost advantage, an explicit systems scope, broader literature review and a frozen validation on '
        'untouched queries or data. This prototype does not establish arbitrary-partition mergeability, broad '
        'distribution-shift robustness, an ML error-prediction improvement, or a production E2E benefit.',
        '', '## Reproducibility', '',
        'See `run.json` for exact input identities, selected source run, implementation hash, settings, package versions '
        'and stage costs. `layout_training_queries.csv` and `layouts.csv` record which training geometries were available '
        'to the selected layouts. `per_query.csv` preserves the estimate, bounds, decision and offline truth for every '
        'candidate. `validity.csv` includes zero/small counts; `interval_audit_all_queries.csv` additionally checks '
        'every generated geometry under every distinct layout, including geometries outside a seed membership. '
        '`metrics.csv` uses true count ≥ 10. '
        '`query_times.csv` preserves resident repetition records. All source payloads are reloaded before summary-only '
        'merge and runtime evaluation.', '',
        'Python: ' + str(state.get('python', 'recorded in run.json')) + '. Package versions: `' +
        json.dumps(state.get('package_versions', {}), sort_keys=True) + '`.', '',
    ]
    if figures:
        report += ['## Figures', '']
        report += [f'![{name}]({name})\n' for name in figures]
    (destination/'analysis_en.md').write_text('\n'.join(report), encoding='utf-8')
    return dict(report='analysis_en.md', figures=figures, development_status=status,
                timing_summary='query_timing_summary.csv', paired_comparison='paired_vs_uniform.csv')

## Run once, preserve previous outputs and export diagnostics even on failure

Run the experiment. The following display cells inspect completed outputs and never retrain or rebuild them.

In [ ]:
def export_results(destination):
    target = destination/'adaptive_bounds_results.zip'
    with ZipFile(target, 'w', ZIP_DEFLATED) as archive:
        for path in sorted(destination.iterdir()):
            if path.is_file() and path.suffix.lower() in ('.csv', '.json', '.md', '.png'):
                archive.write(path, path.name)
    return target


def run_experiment(cfg):
    destination = Path(cfg['output_root']).expanduser()/('adaptive-bounds-'+uuid4().hex[:12])
    destination.mkdir(parents=True, exist_ok=False);start = time.perf_counter()
    state = dict(status='running', study='budgeted-adaptive-bounds', config=cfg, started_utc=datetime.now(timezone.utc).isoformat(),
        implementation_sha256=IMPLEMENTATION_SHA256, python=platform.python_version(), epsilon=EPSILON,
        package_versions={name:importlib.metadata.version(name) for name in ('numpy','pandas','matplotlib')},
        scope='DEVELOPMENT comparison on inspected17; coordinated common layout, not heterogeneous ARM merge or production E2E',
        model_training=False, gate_retuning=False, acceptance_risk_target=None, timings={},
        zero_policy='truth0: error0 only if H0, otherwise infinity; L0<U never certified',
        memory_scope='NumPy buffer bytes are a reproducible lower bound, not total process RSS or cache size')
    save_json(destination/'run.json', state)
    try:
        tick = time.perf_counter();data = load_inputs(cfg, destination, state)
        state['timings']['input_validation_and_loading_seconds'] = time.perf_counter()-tick
        training = freeze_protocol(cfg, data, destination, state)
        candidates = build_candidates(cfg, data, training, destination, state)
        del training
        tick = time.perf_counter();evaluate_candidates(cfg, data, candidates, destination, state)
        state['timings']['evaluation_including_timing_seconds'] = time.perf_counter()-tick
        state.update(status='complete', completed_utc=datetime.now(timezone.utc).isoformat(),
                     total_seconds=time.perf_counter()-start)
        save_json(destination/'run.json', state)
        write_reports(destination, state)
    except Exception as error:
        state.update(status='failed', error=f'{type(error).__name__}: {error}', total_seconds=time.perf_counter()-start)
        save_json(destination/'run.json', state)
        print('Failed run diagnostics:', destination, flush=True)
        raise
    finally:
        archive = export_results(destination)
        print('Results:', destination, '\nShare:', archive, flush=True)
    return destination


if __name__ == '__main__':
    result_dir = run_experiment(load_settings())

## Inspect and export

Share the printed ZIP. `analysis_en.md`, `metrics.csv`, `validity.csv`, `paired_vs_uniform.csv`, `merge_audit.csv`,
`raw_construction.csv`, `costs.csv` and resident timing records support the main interpretation. A failed correctness
audit blocks effectiveness claims. Individual labels/decisions remain available in `per_query.csv`; all candidate
geometries also receive a layout-specific audit in `interval_audit_all_queries.csv`.

In [ ]:
print('Completed result directory:', result_dir)
display(pd.read_csv(result_dir/'metrics.csv'))
display(pd.read_csv(result_dir/'paired_vs_uniform.csv'))

In [ ]:
from IPython.display import Markdown, display as _display
_display(Markdown((result_dir/'analysis_en.md').read_text()))
print('Share:', result_dir/'adaptive_bounds_results.zip')